# FearNetwork MVPA Output Visualization
Ordered report for saved outputs from the `mvpa_L2_*` scripts. The notebook loads checkpoint/intermediate files from `/Users/xiaoqianxiao/projects/NARSAD/MRI/derivatives/fMRI_analysis/LSS/results/FearNetwork` and does not rerun heavy modeling stages.


## Setup


In [ ]:
import os
import math
import glob
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Image, Markdown
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300})

PIPELINE_NAME = "FearNetwork"
PIPELINE_LABEL = "FearNetwork"
OUT_DIR = Path(os.path.join("/Users/xiaoqianxiao/projects/NARSAD/MRI/derivatives/fMRI_analysis/LSS/results", PIPELINE_NAME))
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
LEGACY_CHECKPOINT_DIR = OUT_DIR / "checkpoint"
INTERMEDIATE_DIR = OUT_DIR / "intermediate"
FIGURE_DIR = OUT_DIR / "notebook_figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Pipeline:", PIPELINE_LABEL)
print("OUT_DIR:", OUT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("LEGACY_CHECKPOINT_DIR:", LEGACY_CHECKPOINT_DIR)
print("INTERMEDIATE_DIR:", INTERMEDIATE_DIR)
print("Notebook figures will be saved to:", FIGURE_DIR)


## Helper Functions


In [ ]:
def _existing(paths):
    for path in paths:
        p = Path(path)
        if p.exists():
            return p
    return None


def load_joblib(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        obj = joblib.load(path)
        plt.close("all")
        return obj
    except Exception as exc:
        print(f"Could not load {path}: {exc}")
        plt.close("all")
        return default


def load_first(paths, default=None):
    for path in paths:
        obj = load_joblib(path, default=None)
        if obj is not None:
            print(f"Loaded: {path}")
            return obj
    return default


def coerce_result(obj):
    if obj is None:
        return {}
    if isinstance(obj, dict):
        for key in ["results", "result", "payload", "data"]:
            if key in obj and isinstance(obj[key], dict):
                merged = dict(obj[key])
                for k, v in obj.items():
                    if k not in merged:
                        merged[k] = v
                return merged
        return obj
    return {"value": obj}


def first_present(mapping, keys):
    if mapping is None:
        return None
    for key in keys:
        if isinstance(mapping, dict) and key in mapping and mapping[key] is not None:
            return mapping[key]
    return None


def flatten_result_dict(d, prefix=""):
    rows = []
    if d is None:
        return pd.DataFrame(columns=["key", "value"])
    if not isinstance(d, dict):
        return pd.DataFrame([{"key": prefix or "value", "value": repr(d)}])
    for key, value in d.items():
        name = f"{prefix}.{key}" if prefix else str(key)
        if isinstance(value, dict):
            rows.extend(flatten_result_dict(value, name).to_dict("records"))
        elif isinstance(value, (str, int, float, bool, np.integer, np.floating)) or value is None:
            rows.append({"key": name, "value": value})
        elif isinstance(value, pd.DataFrame):
            rows.append({"key": name, "value": f"DataFrame shape={value.shape}"})
        elif isinstance(value, np.ndarray):
            rows.append({"key": name, "value": f"ndarray shape={value.shape}, dtype={value.dtype}"})
        elif isinstance(value, (list, tuple)):
            rows.append({"key": name, "value": f"{type(value).__name__} len={len(value)}"})
        else:
            rows.append({"key": name, "value": type(value).__name__})
    return pd.DataFrame(rows)


def display_df(df, rows=30):
    if df is None:
        display(Markdown("_No table available._"))
        return
    if isinstance(df, dict):
        df = flatten_result_dict(df)
    if not isinstance(df, pd.DataFrame):
        display(pd.DataFrame({"value": [repr(df)]}))
        return
    if df.empty:
        display(Markdown("_Table is empty._"))
        return
    display(df.head(rows))


def find_figures(*keywords):
    roots = [OUT_DIR, CHECKPOINT_DIR, LEGACY_CHECKPOINT_DIR, INTERMEDIATE_DIR]
    figs = []
    key_l = [k.lower() for k in keywords if k]
    for root in roots:
        if not root.exists():
            continue
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.svg"):
            for path in root.rglob(ext):
                name = path.name.lower()
                if not key_l or any(k in name for k in key_l):
                    figs.append(path)
    return sorted(set(figs))


def show_figures_by_keywords(title, *keywords, max_figs=12):
    figs = find_figures(*keywords)
    display(Markdown(f"**Saved figures: {title}**"))
    if not figs:
        display(Markdown("_No saved figures matched these keywords._"))
        return []
    for path in figs[:max_figs]:
        display(Markdown(f"`{path}`"))
        if path.suffix.lower() == ".svg":
            display(Markdown(f"![{path.name}]({path})"))
        else:
            display(Image(filename=str(path)))
    if len(figs) > max_figs:
        display(Markdown(f"_Showing {max_figs} of {len(figs)} matched figures._"))
    return figs


def save_current_fig(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print(f"Saved {path}")


def plot_matrix(mat, title, labels=None, ax=None, cmap="vlag", center=0):
    arr = np.asarray(mat)
    if arr.ndim == 3:
        arr = np.nanmean(arr, axis=0)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D/3D matrix, got shape {arr.shape}")
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(arr, ax=ax, cmap=cmap, center=center, square=True, annot=True, fmt=".3g", xticklabels=labels, yticklabels=labels, cbar_kws={"shrink": 0.75})
    ax.set_title(title)
    return ax


def maybe_plot_numeric_bars(df, title, max_cols=12):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return False
    numeric = df.select_dtypes(include=[np.number])
    if numeric.empty:
        return False
    means = numeric.mean(numeric_only=True).sort_values(key=lambda s: s.abs(), ascending=False).head(max_cols)
    fig, ax = plt.subplots(figsize=(max(7, len(means) * 0.65), 4))
    sns.barplot(x=means.index, y=means.values, ax=ax, color="#4C78A8")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Mean")
    save_current_fig(title.lower().replace(" ", "_").replace("/", "_") + ".png")
    plt.show()
    return True


def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full(pvals.shape, np.nan, dtype=float)
    valid = np.isfinite(pvals)
    p = pvals[valid]
    if p.size == 0:
        return out
    order = np.argsort(p)
    ranked = p[order] * p.size / np.arange(1, p.size + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    ranked = np.clip(ranked, 0, 1)
    q = np.empty_like(ranked)
    q[order] = ranked
    out[valid] = q
    return out


def welch_by_trial(df, value_col, group_col="Group", trial_col="trial"):
    if df is None or not isinstance(df, pd.DataFrame):
        return pd.DataFrame()
    needed = {value_col, group_col, trial_col}
    if not needed.issubset(df.columns):
        return pd.DataFrame()
    rows = []
    for trial, sub in df.groupby(trial_col):
        sad = pd.to_numeric(sub.loc[sub[group_col].astype(str).str.upper().eq("SAD"), value_col], errors="coerce").dropna()
        hc = pd.to_numeric(sub.loc[sub[group_col].astype(str).str.upper().eq("HC"), value_col], errors="coerce").dropna()
        if len(sad) >= 2 and len(hc) >= 2:
            t, p = stats.ttest_ind(sad, hc, equal_var=False, nan_policy="omit")
            pooled = math.sqrt(((sad.std(ddof=1) ** 2) + (hc.std(ddof=1) ** 2)) / 2) if len(sad) > 1 and len(hc) > 1 else np.nan
            d = (sad.mean() - hc.mean()) / pooled if pooled and np.isfinite(pooled) and pooled > 0 else np.nan
        else:
            t, p, d = np.nan, np.nan, np.nan
        rows.append({"trial": trial, "n_sad": len(sad), "n_hc": len(hc), "mean_sad": sad.mean() if len(sad) else np.nan, "mean_hc": hc.mean() if len(hc) else np.nan, "t": t, "p": p, "cohens_d": d})
    out = pd.DataFrame(rows).sort_values("trial") if rows else pd.DataFrame()
    if not out.empty:
        out["p_fdr_bh"] = bh_fdr(out["p"].to_numpy())
    return out


def extract_first_dataframe(result, preferred=()):
    if not isinstance(result, dict):
        return None
    for key in preferred:
        val = result.get(key)
        if isinstance(val, pd.DataFrame):
            return val
    for val in result.values():
        if isinstance(val, pd.DataFrame):
            return val
    return None


def plot_corr_table(df, title):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        display(Markdown("_No correlation table available._"))
        return
    display_df(df, rows=40)
    possible_r = [c for c in df.columns if c.lower() in {"r", "rho", "corr", "correlation", "estimate"}]
    x_cols = [c for c in df.columns if any(s in c.lower() for s in ["neural", "metric", "index", "predictor"])]
    y_cols = [c for c in df.columns if any(s in c.lower() for s in ["clinical", "score", "outcome", "measure"])]
    if possible_r and x_cols and y_cols:
        try:
            piv = df.pivot_table(index=y_cols[0], columns=x_cols[0], values=possible_r[0], aggfunc="mean")
            fig, ax = plt.subplots(figsize=(max(7, 0.45 * piv.shape[1]), max(4, 0.35 * piv.shape[0])))
            sns.heatmap(piv, cmap="vlag", center=0, ax=ax, cbar_kws={"label": possible_r[0]})
            ax.set_title(title)
            save_current_fig(title.lower().replace(" ", "_").replace("/", "_") + ".png")
            plt.show()
        except Exception as exc:
            print(f"Could not make correlation heatmap: {exc}")


## Output Manifest


In [ ]:
# Load saved stage/checkpoint payloads. Missing files are expected if a stage has not run yet.
paths = {
    "stage03_data": [CHECKPOINT_DIR / "cell_03.joblib", LEGACY_CHECKPOINT_DIR / "cell_03.joblib"],
    "stage06_results_11": [CHECKPOINT_DIR / "results_analysis_11.joblib", CHECKPOINT_DIR / "results_11.joblib", CHECKPOINT_DIR / "cell_06.joblib", LEGACY_CHECKPOINT_DIR / "results_analysis_11.joblib", LEGACY_CHECKPOINT_DIR / "results_11.joblib", LEGACY_CHECKPOINT_DIR / "cell_06.joblib"],
    "stage10_haufe": [INTERMEDIATE_DIR / "stage10_haufe_maps.joblib", CHECKPOINT_DIR / "cell_10.joblib", LEGACY_CHECKPOINT_DIR / "cell_10.joblib"],
    "stage11_importance": [INTERMEDIATE_DIR / "stage11_importance_masks.joblib", CHECKPOINT_DIR / "cell_11.joblib"],
    "stage12_topology": [CHECKPOINT_DIR / "results_analysis_12.joblib", INTERMEDIATE_DIR / "stage12_topology.joblib", CHECKPOINT_DIR / "cell_12.joblib"],
    "stage13_drift": [CHECKPOINT_DIR / "results_analysis_13.joblib", INTERMEDIATE_DIR / "stage13_drift.joblib", CHECKPOINT_DIR / "cell_13.joblib"],
    "stage14_trajectories": [CHECKPOINT_DIR / "results_analysis_13_trajectories.joblib", INTERMEDIATE_DIR / "stage14_trajectories.joblib", CHECKPOINT_DIR / "cell_14.joblib"],
    "stage15_decision": [CHECKPOINT_DIR / "results_analysis_14.joblib", INTERMEDIATE_DIR / "stage15_decision_boundary.joblib", CHECKPOINT_DIR / "cell_15.joblib"],
    "stage16_safety_threat": [CHECKPOINT_DIR / "cell_24.joblib", CHECKPOINT_DIR / "cell_14.joblib", CHECKPOINT_DIR / "results_analysis_21.joblib", INTERMEDIATE_DIR / "stage16_safety_threat.joblib"],
    "stage18_drift_efficiency": [CHECKPOINT_DIR / "cell_24.joblib", CHECKPOINT_DIR / "cell_15.joblib", CHECKPOINT_DIR / "results_analysis_22.joblib", INTERMEDIATE_DIR / "stage18_drift_efficiency.joblib"],
    "stage19_prob_opening": [CHECKPOINT_DIR / "cell_16_opening_test.joblib", CHECKPOINT_DIR / "cell_16.joblib", CHECKPOINT_DIR / "results_analysis_23.joblib", INTERMEDIATE_DIR / "stage19_probabilistic_opening.joblib"],
    "stage20_realignment": [CHECKPOINT_DIR / "cell_17_realignment.joblib", CHECKPOINT_DIR / "results_analysis_24.joblib", INTERMEDIATE_DIR / "stage20_spatial_realignment.joblib", CHECKPOINT_DIR / "cell_20.joblib"],
    "stage21_reverse": [CHECKPOINT_DIR / "cell_18_reverse_cross_decoding.joblib", CHECKPOINT_DIR / "cell_18_realignment.joblib", CHECKPOINT_DIR / "results_analysis_25.joblib", INTERMEDIATE_DIR / "stage21_reverse_cross_decoding.joblib", CHECKPOINT_DIR / "cell_21.joblib"],
    "stage23_clinical_scores": [INTERMEDIATE_DIR / "stage23_clinical_scores.joblib", CHECKPOINT_DIR / "cell_23.joblib"],
    "stage24_neural_indices": [INTERMEDIATE_DIR / "stage24_neural_clinical_indices.joblib", CHECKPOINT_DIR / "cell_24.joblib"],
    "stage26_master_merge": [CHECKPOINT_DIR / "cell_26.joblib", INTERMEDIATE_DIR / "stage26_master_neural_clinical.joblib"],
    "stage27_pearson": [INTERMEDIATE_DIR / "stage27_neural_clinical_pearson.joblib", CHECKPOINT_DIR / "cell_27.joblib"],
    "stage28_partial": [INTERMEDIATE_DIR / "stage28_neural_clinical_partial.joblib", CHECKPOINT_DIR / "cell_28.joblib"],
    "stage29_zscore_outlier": [INTERMEDIATE_DIR / "stage29_zscore_outlier.joblib", CHECKPOINT_DIR / "cell_29.joblib"],
    "stage30_ols": [INTERMEDIATE_DIR / "stage30_z_ols_regression.joblib", CHECKPOINT_DIR / "cell_30.joblib"],
}
loaded = {name: load_first(candidates, default=None) for name, candidates in paths.items()}
results = {name: coerce_result(payload) for name, payload in loaded.items()}
manifest = []
for name, candidates in paths.items():
    existing = _existing(candidates)
    manifest.append({"stage": name, "found": existing is not None, "path": str(existing) if existing else ""})
display_df(pd.DataFrame(manifest), rows=50)


## 1. Analysis 1.1: Neural Dissociation Self-Decoding
This section summarizes self-decoding accuracy with null distribution, functional specificity, and spatial specificity where those outputs were saved by the script.


In [ ]:
r11_payload = results.get("stage06_results_11", {})
r11 = r11_payload.get("results_11", r11_payload) if isinstance(r11_payload, dict) else {}
display_df(flatten_result_dict(r11), rows=60)

# Recreate the four-panel Analysis 1.1 figure: self-decoding null distributions, functional specificity, and spatial specificity.
def _lookup_many(*keys):
    for key in keys:
        if isinstance(r11, dict) and key in r11 and r11[key] is not None:
            return r11[key]
        if isinstance(r11_payload, dict) and key in r11_payload and r11_payload[key] is not None:
            return r11_payload[key]
    return None


def _scalar_or_nan(value):
    try:
        arr = np.asarray(value, dtype=float)
        if arr.size == 1:
            return float(arr.ravel()[0])
    except Exception:
        pass
    return np.nan


def _format_p(p):
    p = _scalar_or_nan(p)
    if np.isfinite(p):
        return f"p = {p:.4f}"
    return "p = n/a"


def _plot_self_decoding(ax, null_dist, obs, p_val, title):
    obs = _scalar_or_nan(obs)
    null_arr = np.asarray(null_dist, dtype=float).ravel() if null_dist is not None else np.array([])
    null_arr = null_arr[np.isfinite(null_arr)]
    if null_arr.size:
        sns.histplot(null_arr, stat="density", bins=35, color="lightgray", edgecolor="black", alpha=0.85, ax=ax, label="Null Dist")
        try:
            sns.kdeplot(null_arr, ax=ax, color="gray", linewidth=2.5)
        except Exception:
            pass
        thresh = np.nanpercentile(null_arr, 95)
        ax.axvline(thresh, color="blue", linestyle="--", linewidth=2, label="95% null")
    else:
        ax.text(0.5, 0.5, "Null distribution not found", ha="center", va="center", transform=ax.transAxes)
    if np.isfinite(obs):
        ax.axvline(obs, color="red", linewidth=2.5, label=f"Obs: {obs:.2f}")
    ax.set_title(f"{title} (CV Acc: {obs:.2f})\n({_format_p(p_val)})" if np.isfinite(obs) else f"{title}\n({_format_p(p_val)})")
    ax.set_xlabel("Forced-choice accuracy")
    ax.set_ylabel("Density")
    ax.legend(frameon=True, fontsize=10)

acc_sad = _lookup_many("acc_sad_cv", "accuracy_sad", "sad_accuracy")
acc_hc = _lookup_many("acc_hc_cv", "accuracy_hc", "hc_accuracy")
p_sad = _lookup_many("p_sad", "p_acc_sad", "sad_p")
p_hc = _lookup_many("p_hc", "p_acc_hc", "hc_p")
perm_sad = _lookup_many("perm_dist_sad", "perm_acc_sad", "null_dist_sad", "sad_null")
perm_hc = _lookup_many("perm_dist_hc", "perm_acc_hc", "null_dist_hc", "hc_null")
func_matrix = _lookup_many("func_matrix", "functional_specificity", "functional_matrix")
func_pvals = _lookup_many("p_func_pvals", "func_pvals", "functional_pvals")
sim_spatial = _lookup_many("sim_spatial", "obs_sim", "patterm_similarity", "pattern_similarity")
p_sim = _lookup_many("p_sim", "p_sim_spatial", "p_spatial")
spatial_matrix = _lookup_many("spatial_matrix")
spatial_pvals = _lookup_many("spatial_pvals")

if func_matrix is None:
    func_matrix = np.array([[_scalar_or_nan(acc_sad), np.nan], [np.nan, _scalar_or_nan(acc_hc)]], dtype=float)
else:
    func_matrix = np.asarray(func_matrix, dtype=float)
if func_pvals is None:
    func_pvals = np.array([[_scalar_or_nan(p_sad), np.nan], [np.nan, _scalar_or_nan(p_hc)]], dtype=float)
else:
    func_pvals = np.asarray(func_pvals, dtype=float)
if spatial_matrix is None:
    sim = _scalar_or_nan(sim_spatial)
    spatial_matrix = np.array([[1.0, sim], [sim, 1.0]], dtype=float)
else:
    spatial_matrix = np.asarray(spatial_matrix, dtype=float)
if spatial_pvals is None:
    ps = _scalar_or_nan(p_sim)
    spatial_pvals = np.array([[0.0, ps], [ps, 0.0]], dtype=float)
else:
    spatial_pvals = np.asarray(spatial_pvals, dtype=float)

if any(x is not None for x in [perm_sad, perm_hc, func_matrix, spatial_matrix]):
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.25], hspace=0.38, wspace=0.28)
    _plot_self_decoding(fig.add_subplot(gs[0, 0]), perm_sad, acc_sad, p_sad, "SAD Self-Decoding")
    _plot_self_decoding(fig.add_subplot(gs[0, 1]), perm_hc, acc_hc, p_hc, "HC Self-Decoding")
    annot_func = np.empty(func_matrix.shape, dtype=object)
    for i in range(func_matrix.shape[0]):
        for j in range(func_matrix.shape[1]):
            val = func_matrix[i, j]
            pv = func_pvals[i, j] if func_pvals.shape == func_matrix.shape else np.nan
            star = "*" if np.isfinite(pv) and pv < 0.05 else ""
            annot_func[i, j] = f"{val:.3f}\n({star})" if np.isfinite(val) else "n/a"
    ax3 = fig.add_subplot(gs[1, 0])
    sns.heatmap(func_matrix, annot=annot_func, fmt="", cmap="RdBu_r", center=0.5, vmin=0.3, vmax=0.9, cbar=True, xticklabels=["Test SAD", "Test HC"], yticklabels=["Train SAD", "Train HC"], ax=ax3)
    ax3.set_title("Functional Specificity\n(Forced-Choice Accuracy)")
    annot_spatial = np.empty(spatial_matrix.shape, dtype=object)
    for i in range(spatial_matrix.shape[0]):
        for j in range(spatial_matrix.shape[1]):
            val = spatial_matrix[i, j]
            pv = spatial_pvals[i, j] if spatial_pvals.shape == spatial_matrix.shape else np.nan
            star = "*" if i != j and np.isfinite(pv) and pv < 0.05 else ""
            annot_spatial[i, j] = f"{val:.3f}\n{star}" if np.isfinite(val) else "n/a"
    ax4 = fig.add_subplot(gs[1, 1])
    sns.heatmap(spatial_matrix, annot=annot_spatial, fmt="", cmap="RdBu_r", center=0, vmin=-1, vmax=1, cbar=True, xticklabels=["SAD Map", "HC Map"], yticklabels=["SAD Map", "HC Map"], ax=ax4)
    ax4.set_title("Spatial Specificity")
    save_current_fig("section01_neural_dissociation_four_panel.png")
    plt.show()
else:
    display(Markdown("_Analysis 1.1 four-panel ingredients were not found in the saved payload._"))
    show_figures_by_keywords("Saved Analysis 1.1 fallback", "analysis_11", "results_11", "neural_dissociation", "functional_specificity", "spatial_specificity", max_figs=4)

for key in ["acc_summary", "accuracy_summary", "df_accuracy", "df_acc", "functional_specificity", "spatial_specificity", "null_distribution"]:
    val = r11.get(key) if isinstance(r11, dict) else None
    if isinstance(val, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(val, rows=30)
    elif isinstance(val, np.ndarray) and val.ndim in (1, 2):
        fig, ax = plt.subplots(figsize=(6, 4))
        if val.ndim == 1:
            ax.hist(val[np.isfinite(val)], bins=30, color="#4C78A8", alpha=0.8)
            ax.set_title(key)
        else:
            plot_matrix(val, key, ax=ax)
        save_current_fig(f"section01_{key}.png")
        plt.show()


## 2. Haufe / Z-Scored Maps
Glass-brain figures are shown if the scripts saved them. The table lists available Haufe or z-map payloads for traceability.


In [ ]:
r10 = results.get("stage10_haufe", {})
r03 = results.get("stage03_data", {})
r11_payload = results.get("stage06_results_11", {})
r11_for_maps = r11_payload.get("results_11", r11_payload) if isinstance(r11_payload, dict) else {}
display_df(flatten_result_dict(r10), rows=60)

# FearNetworkAll/MemoryFearNetwork-style glass-brain reconstruction for ROI permutation maps.
def _nested_get(mapping, keys):
    if not isinstance(mapping, dict):
        return None
    for key in keys:
        if key in mapping and mapping[key] is not None:
            return mapping[key]
    for value in mapping.values():
        if isinstance(value, dict):
            found = _nested_get(value, keys)
            if found is not None:
                return found
    return None


def _pipeline_roi_candidates():
    if PIPELINE_NAME == "FearNetwork":
        roots = [
            Path("/Users/xiaoqianxiao/tool/parcellation/Gillian_anatomically_constrained"),
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI/Gillian_anatomically_constrained"),
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs/NARSAD_uni/Gillian_NARSAD"),
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs/FearNetwork"),
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI/FearNetwork"),
        ]
    elif PIPELINE_NAME == "MemoryFearNetwork":
        roots = [
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs/MemoryFearNetwork"),
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI/MemoryFearNetwork"),
            Path("/Users/xiaoqianxiao/tool/parcellation/Gillian_anatomically_constrained"),
        ]
    else:
        roots = [
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI") / PIPELINE_NAME,
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs") / PIPELINE_NAME,
            Path("/Users/xiaoqianxiao/tool/parcellation/Gillian_anatomically_constrained"),
        ]
    env_name = "MEMORY_FEAR_ROI_DIR" if PIPELINE_NAME == "MemoryFearNetwork" else "FEAR_ROI_DIR"
    env_path = os.environ.get(env_name) or os.environ.get("ROI_DIR")
    if env_path:
        roots.insert(0, Path(env_path))
    return [p for p in roots if p.exists()]


def _default_roi_order():
    rois = _nested_get(r03, ["TARGET_ROIS", "ROI_ORDER", "roi_order", "target_rois"])
    if rois is not None:
        return list(rois)
    if PIPELINE_NAME == "MemoryFearNetwork":
        return ["left_acc", "left_amygdala", "left_hippocampus", "left_insula", "left_vmpfc", "left_VVC", "left_AG", "left_SMG", "left_IFG", "left_MFG", "left_SFG", "left_Precuneus", "right_acc", "right_amygdala", "right_hippocampus", "right_insula", "right_vmpfc", "right_VVC", "right_AG", "right_SMG", "right_IFG", "right_MFG", "right_SFG", "right_Precuneus"]
    return ["left_acc", "left_amygdala", "left_hippocampus", "left_insula", "left_vmpfc", "right_acc", "right_amygdala", "right_hippocampus", "right_insula", "right_vmpfc"]


def _find_roi_dir(roi_order):
    for roi_dir in _pipeline_roi_candidates():
        if all(list(roi_dir.glob(f"*{roi}*.nii*")) for roi in roi_order):
            return roi_dir
    return None


def _roi_chunk_lengths(roi_order, roi_dir):
    try:
        import nibabel as nib
    except Exception:
        return None, None, "nibabel unavailable"
    chunks = []
    ref_img = None
    for roi in roi_order:
        files = sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))
        if not files:
            return None, None, f"missing mask for {roi} in {roi_dir}"
        img = nib.load(str(files[0]))
        if ref_img is None:
            ref_img = img
        chunks.append(int(np.sum(img.get_fdata() > 0)))
    return chunks, ref_img, None


def _bh_mask(pvals, alpha=0.05):
    p = np.asarray(pvals, dtype=float).ravel()
    valid = np.isfinite(p)
    out = np.zeros(p.shape, dtype=bool)
    if valid.sum() == 0:
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    thresh = alpha * np.arange(1, len(ranked) + 1) / len(ranked)
    passed = ranked <= thresh
    if passed.any():
        cutoff = ranked[np.where(passed)[0].max()]
        out[valid] = pv <= cutoff
    return out


def _roi_fdr_mask(pvals, roi_order, roi_dir, alpha=0.05):
    chunks, _, err = _roi_chunk_lengths(roi_order, roi_dir)
    p = np.asarray(pvals, dtype=float).ravel()
    if chunks is None or sum(chunks) != p.size:
        return _bh_mask(p, alpha=alpha), "Global FDR fallback"
    mask = np.zeros(p.shape, dtype=bool)
    cursor = 0
    for n in chunks:
        mask[cursor: cursor + n] = _bh_mask(p[cursor: cursor + n], alpha=alpha)
        cursor += n
    return mask, "ROI-FDR"


def _load_perm_cache(group):
    candidates = [
        CHECKPOINT_DIR / f"perm_results_{group}_fear_network_2way.joblib",
        LEGACY_CHECKPOINT_DIR / f"perm_results_{group}_fear_network_2way.joblib",
        OUT_DIR / f"perm_results_{group}_fear_network_2way.joblib",
        CHECKPOINT_DIR / f"perm_results_{group}_wholeBrain_voxelwise.joblib",
        LEGACY_CHECKPOINT_DIR / f"perm_results_{group}_wholeBrain_voxelwise.joblib",
        OUT_DIR / f"perm_results_{group}_wholeBrain_voxelwise.joblib",
    ]
    for path in candidates:
        obj = load_joblib(path, default=None)
        if isinstance(obj, dict) and {"obs_weights", "null_weights", "p_values_raw"}.issubset(obj.keys()):
            return obj, path
    return None, None


def _z_from_cache(cache):
    obs = np.asarray(cache["obs_weights"], dtype=float).ravel()
    null = np.asarray(cache["null_weights"], dtype=float)
    z = (obs - np.nanmean(null, axis=0)) / (np.nanstd(null, axis=0) + 1e-12)
    z = np.asarray(z, dtype=float).ravel()
    # The source notebooks flip CSR-first models so CSS/threat-positive evidence is red.
    if PIPELINE_NAME in {"FearNetwork", "MemoryFearNetwork"}:
        z = -z
    return z



def _roi_selection_stats(sig_mask, roi_order, roi_dir, group, mode):
    chunks, _, err = _roi_chunk_lengths(roi_order, roi_dir)
    if chunks is None:
        return pd.DataFrame([{"Group": group, "Mode": mode, "ROI": "ROI stats unavailable", "ROI_voxels": np.nan, "Selected_voxels": int(np.sum(sig_mask)), "Percent_ROI": np.nan, "Note": err}])
    rows = []
    cursor = 0
    sig_arr = np.asarray(sig_mask, dtype=bool).ravel()
    for roi, n in zip(roi_order, chunks):
        roi_mask = sig_arr[cursor: cursor + n]
        selected = int(np.sum(roi_mask))
        rows.append({"Group": group, "Mode": mode, "ROI": roi, "ROI_voxels": int(n), "Selected_voxels": selected, "Percent_ROI": 100.0 * selected / n if n else np.nan, "Note": ""})
        cursor += n
    return pd.DataFrame(rows)

def _matched_perm_maps():
    roi_order = _default_roi_order()
    roi_dir = _find_roi_dir(roi_order)
    caches = {}
    for group in ["SAD", "HC"]:
        cache, path = _load_perm_cache(group)
        if cache is not None:
            caches[group] = {"cache": cache, "path": path, "z": _z_from_cache(cache), "p": np.asarray(cache["p_values_raw"], dtype=float).ravel()}
    if set(caches) != {"SAD", "HC"}:
        return {}, [f"Permutation caches found for groups: {sorted(caches)}"]
    fdr_masks = {}
    fdr_counts = {}
    mode0 = {}
    for group, payload in caches.items():
        if roi_dir is not None:
            fdr_masks[group], mode0[group] = _roi_fdr_mask(payload["p"], roi_order, roi_dir)
        else:
            fdr_masks[group], mode0[group] = _bh_mask(payload["p"]), "Global FDR; ROI masks not found"
        fdr_counts[group] = int(np.sum(fdr_masks[group]))
    if fdr_counts["SAD"] > 0 and fdr_counts["HC"] > 0:
        match_cfg = {"SAD": None, "HC": None}
    elif fdr_counts["SAD"] > 0 and fdr_counts["HC"] == 0:
        match_cfg = {"SAD": None, "HC": fdr_counts["SAD"]}
    elif fdr_counts["HC"] > 0 and fdr_counts["SAD"] == 0:
        match_cfg = {"SAD": fdr_counts["HC"], "HC": None}
    else:
        match_cfg = {"SAD": "fallback", "HC": "fallback"}
    maps = {}
    messages = [f"ROI_DIR={roi_dir}", f"Initial ROI-FDR counts: SAD={fdr_counts['SAD']}, HC={fdr_counts['HC']}"]
    for group, payload in caches.items():
        z = payload["z"]
        cfg = match_cfg[group]
        if cfg is None:
            sig_mask = fdr_masks[group]
            mode = mode0[group]
        elif cfg == "fallback":
            cutoff = np.nanpercentile(np.abs(z[np.isfinite(z)]), 98)
            sig_mask = np.isfinite(z) & (np.abs(z) >= cutoff)
            mode = "Top 2% Fallback"
        else:
            top_idx = np.argsort(np.abs(z))[-int(cfg):]
            sig_mask = np.zeros(z.shape, dtype=bool)
            sig_mask[top_idx] = True
            mode = f"Top {int(cfg)} (Matched)"
        z_masked = z * sig_mask
        display_mask = sig_mask.copy()
        roi_stats = _roi_selection_stats(display_mask, roi_order, roi_dir, group, mode) if roi_dir is not None else pd.DataFrame()
        maps[group] = {"values": z_masked, "mode": mode, "path": payload["path"], "n": int(np.sum(display_mask)), "selection_n_before_direction_filter": int(np.sum(sig_mask)), "roi_order": roi_order, "roi_dir": roi_dir, "roi_stats": roi_stats}
    return maps, messages


def _as_img_like(obj):
    if obj is None:
        return None
    try:
        import nibabel as nib
        from nilearn import image
        if isinstance(obj, nib.spatialimages.SpatialImage):
            return obj
        if isinstance(obj, (str, Path)) and Path(obj).exists():
            return image.load_img(str(obj))
    except Exception:
        return None
    return None


def _reconstruct_roi_map(flat_data, roi_order, roi_dir):
    try:
        import nibabel as nib
    except Exception as exc:
        return None, f"nibabel unavailable: {exc}"
    chunks, ref_img, err = _roi_chunk_lengths(roi_order, roi_dir)
    if chunks is None:
        return None, err
    arr = np.asarray(flat_data, dtype=float).ravel()
    if sum(chunks) != arr.size:
        return None, f"ROI mask voxel count {sum(chunks)} does not match map length {arr.size}"
    final_vol = np.zeros(ref_img.shape, dtype=float)
    cursor = 0
    for roi, n in zip(roi_order, chunks):
        files = sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))
        mask_img = nib.load(str(files[0]))
        mask_data = mask_img.get_fdata() > 0
        final_vol[mask_data] = arr[cursor: cursor + n]
        cursor += n
    return nib.Nifti1Image(final_vol, ref_img.affine, ref_img.header), None


def _vector_to_img(values, map_info=None):
    try:
        import nibabel as nib
        from nilearn import masking
    except Exception as exc:
        return None, f"nilearn/nibabel unavailable: {exc}"
    img = _as_img_like(values)
    if img is not None:
        return img, None
    arr = np.asarray(values, dtype=float).ravel() if values is not None else np.array([])
    if arr.size == 0:
        return None, "map vector not found"
    if isinstance(map_info, dict) and map_info.get("roi_dir") is not None:
        img, err = _reconstruct_roi_map(arr, map_info["roi_order"], map_info["roi_dir"])
        if img is not None:
            return img, None
    mask_source = _nested_get(r10, ["mask_img", "brain_mask", "mask_ext", "roi_mask", "wholebrain_mask"])
    if mask_source is None:
        mask_source = _nested_get(r11_for_maps, ["mask_img", "brain_mask", "mask_ext", "roi_mask", "wholebrain_mask"])
    mask_img = _as_img_like(mask_source)
    if mask_img is not None:
        try:
            if int(mask_img.get_fdata().astype(bool).sum()) == arr.size:
                return masking.unmask(arr, mask_img), None
        except Exception as exc:
            return None, f"mask unmask failed: {exc}"
    return None, f"map has {arr.size} values but no compatible ROI masks or whole-brain mask were found"


def _stage_map(group):
    g = group.lower()
    candidates = [f"z_map_{g}", f"z_scores_{g}", f"z_haufe_{g}", f"haufe_z_{g}", f"map_{g}_z", f"map_{g}", f"haufe_{g}", f"pattern_{g}"]
    found = _nested_get(r10, candidates)
    if found is None:
        found = _nested_get(r11_for_maps, candidates)
    return {"values": found, "mode": "Haufe/Z map from stage output", "roi_order": _default_roi_order(), "roi_dir": _find_roi_dir(_default_roi_order())}


def _plot_glass(group, map_info, ax=None):
    try:
        from nilearn import plotting
    except Exception as exc:
        return False, f"nilearn plotting unavailable: {exc}"
    img, err = _vector_to_img(map_info.get("values"), map_info)
    if img is None:
        return False, err
    data = np.asarray(img.get_fdata(), dtype=float)
    finite = data[np.isfinite(data)]
    finite_nonzero = finite[finite != 0]
    if finite_nonzero.size == 0:
        return False, "map image contains no non-zero finite values"
    vmax = float(np.nanpercentile(np.abs(finite_nonzero), 99))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = float(np.nanmax(np.abs(finite_nonzero)))
    threshold = float(np.nanmin(np.abs(finite_nonzero)))
    cmap = "RdBu_r"
    title = f"{group} Fear Network: {map_info.get('mode')} (all selected directions)" if PIPELINE_NAME == "FearNetwork" else f"{group} {PIPELINE_LABEL}: {map_info.get('mode')}"
    plotting.plot_glass_brain(img, display_mode="lyrz", colorbar=True, threshold=threshold, vmax=vmax, cmap=cmap, plot_abs=False, black_bg=False, axes=ax, title=title)
    return True, map_info.get("mode")

matched_maps, match_messages = _matched_perm_maps()
if matched_maps:
    fig = plt.figure(figsize=(10, 8))
    axes = {"SAD": fig.add_subplot(2, 1, 1), "HC": fig.add_subplot(2, 1, 2)}
    messages = []
    rendered = []
    for group, ax in axes.items():
        ok, msg = _plot_glass(group, matched_maps[group], ax=ax)
        rendered.append(ok)
        if not ok:
            ax.axis("off")
            ax.text(0.5, 0.5, f"{group} glass brain could not be reconstructed\n{msg}", ha="center", va="center", wrap=True)
            messages.append(f"{group}: {msg}")
    save_current_fig("section02_fearnetwork_roi_fdr_glass_brain.png")
    plt.show()
    if messages:
        display(Markdown("; ".join(messages)))
    group_summary = pd.DataFrame([{ "group": g, "mode": v["mode"], "selected_voxels_shown": v["n"], "selected_before_direction_filter": v.get("selection_n_before_direction_filter", v["n"]), "cache": str(v["path"]), "roi_dir": str(v["roi_dir"]) } for g, v in matched_maps.items()])
    display_df(group_summary, rows=10)
    roi_plot_frames = []
    for group in ["SAD", "HC"]:
        roi_stats = matched_maps[group].get("roi_stats")
        mode = matched_maps[group].get("mode", "")
        print(f"\nResults for {group} ({mode}):")
        print(f"{'ROI Name':<25} | {'Voxels':<8} | {'% ROI'}")
        print("-" * 50)
        if isinstance(roi_stats, pd.DataFrame) and not roi_stats.empty:
            roi_print = roi_stats[roi_stats["Selected_voxels"] > 0].sort_values("Selected_voxels", ascending=False)
            if roi_print.empty:
                print("No displayed significant voxels.")
            for _, row in roi_print.iterrows():
                print(f"{row['ROI']:<25} | {int(row['Selected_voxels']):<8} | {row['Percent_ROI']:.2f}%")
            roi_plot_frames.append(roi_stats.copy())
        else:
            print("ROI statistics unavailable.")
    if roi_plot_frames:
        roi_plot = pd.concat(roi_plot_frames, ignore_index=True)
        roi_plot = roi_plot[roi_plot["Selected_voxels"] > 0].copy()
        if not roi_plot.empty:
            requested_roi_order = ["left_hippocampus", "right_hippocampus", "left_insula", "right_insula", "left_acc", "right_acc", "left_vmpfc", "right_vmpfc", "left_amygdala", "right_amygdala"]
            roi_order_plot = [roi for roi in requested_roi_order if roi in set(roi_plot["ROI"])]
            extra_rois = [roi for roi in roi_plot.groupby("ROI")["Selected_voxels"].sum().sort_values(ascending=False).index.tolist() if roi not in roi_order_plot]
            roi_order_plot = roi_order_plot + extra_rois
            fig_roi, ax_roi = plt.subplots(figsize=(11, 5))
            sns.barplot(data=roi_plot, x="ROI", y="Percent_ROI", hue="Group", order=roi_order_plot, palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=ax_roi)
            for container in ax_roi.containers:
                labels = []
                for bar in container:
                    height = bar.get_height()
                    labels.append(f"{height:.1f}%" if np.isfinite(height) and height > 0 else "")
                ax_roi.bar_label(container, labels=labels, padding=3, fontsize=9, rotation=90)
            ax_roi.set_ylim(0, max(roi_plot["Percent_ROI"].max() * 1.22, 1))
            ax_roi.set_ylabel("Selected voxels (% of ROI)")
            ax_roi.set_xlabel("")
            ax_roi.set_title("ROI distribution of selected significant voxels")
            ax_roi.tick_params(axis='x', rotation=60)
            save_current_fig("section02_roi_percent_sig_voxels.png")
            plt.show()
    display(Markdown("; ".join(match_messages)))
else:
    display(Markdown("_Permutation-cache ROI glass brain was unavailable; falling back to stage Haufe/Z maps._"))
    display(Markdown("; ".join(match_messages)))
    fig = plt.figure(figsize=(10, 8))
    axes = {"SAD": fig.add_subplot(2, 1, 1), "HC": fig.add_subplot(2, 1, 2)}
    rendered = []
    messages = []
    for group, ax in axes.items():
        ok, msg = _plot_glass(group, _stage_map(group), ax=ax)
        rendered.append(ok)
        if not ok:
            ax.axis("off")
            ax.text(0.5, 0.5, f"{group} glass brain could not be reconstructed\n{msg}", ha="center", va="center", wrap=True)
            messages.append(f"{group}: {msg}")
    if any(rendered):
        save_current_fig("section02_haufe_z_glass_brain.png")
        plt.show()
    else:
        plt.close(fig)
        display(Markdown("_Could not reconstruct the Haufe/Z glass-brain figure from saved arrays in this local notebook._"))
        display(Markdown("Missing/issue details: " + "; ".join(messages)))
        show_figures_by_keywords("Saved Haufe/glass-brain fallback", "haufe", "zscore", "z_score", "glass", "activation_pattern", max_figs=4)


## 3. Permutation-Significant Voxels
This section reports permutation-importance feature-mask diagnostics and any saved glass-brain/ribbon figures for significant or fallback feature spaces.


In [ ]:
r11imp = results.get("stage11_importance", {})
display_df(flatten_result_dict(r11imp), rows=80)

def _stage11_dict(key_options):
    for key in key_options:
        val = r11imp.get(key) if isinstance(r11imp, dict) else None
        if isinstance(val, dict):
            return val
    return {}

masks = _stage11_dict(["importance_mask_permutated", "importance_masks_permutated", "importance_masks", "mask"])
scores = _stage11_dict(["importance_scores_permutated", "importance_scores", "scores", "actual_importance"])
pvals = _stage11_dict(["p_values_permutated", "p_values", "pvals"])
rows = []
for group in ["SAD", "HC"]:
    mask = masks.get(group)
    score = scores.get(group)
    pval = pvals.get(group)
    rows.append({
        "group": group,
        "mask_features": int(np.asarray(mask, dtype=bool).sum()) if mask is not None else np.nan,
        "positive_importance": int((np.asarray(score) > 0).sum()) if score is not None else np.nan,
        "max_importance": float(np.nanmax(score)) if score is not None and np.asarray(score).size else np.nan,
        "min_p": float(np.nanmin(pval)) if pval is not None and np.asarray(pval).size else np.nan,
    })
summary = pd.DataFrame(rows)
display_df(summary, rows=10)
if not summary.empty:
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.barplot(data=summary, x="group", y="mask_features", ax=ax, color="#59A14F")
    ax.set_title("Permutation-significant feature counts")
    ax.set_ylabel("Selected voxels")
    save_current_fig("section03_permutation_feature_counts.png")
    plt.show()

# Glass brain for all permutation-significant voxels from stage 11.
def _roi_vector_order_fear():
    # Stage 11 feature vectors follow the original ROI concatenation order from the L2 script.
    return ["left_acc", "left_amygdala", "left_hippocampus", "left_insula", "left_vmpfc", "right_acc", "right_amygdala", "right_hippocampus", "right_insula", "right_vmpfc"]

def _roi_plot_order_fear():
    return ["left_hippocampus", "right_hippocampus", "left_insula", "right_insula", "left_acc", "right_acc", "left_vmpfc", "right_vmpfc", "left_amygdala", "right_amygdala"]

def _roi_dir_fear():
    candidates = [
        Path("/Users/xiaoqianxiao/tool/parcellation/Gillian_anatomically_constrained"),
        Path("/Users/xiaoqianxiao/projects/NARSAD/ROI/Gillian_anatomically_constrained"),
        Path("/Users/xiaoqianxiao/tool/parcellation/ROIs/NARSAD_uni/Gillian_NARSAD"),
    ]
    for d in candidates:
        if d.exists() and all(list(d.glob(f"*{roi}*.nii*")) for roi in _roi_vector_order_fear()):
            return d
    return None

def _reconstruct_roi_map(flat_data, roi_order, roi_dir):
    try:
        import nibabel as nib
    except Exception as exc:
        return None, f"nibabel unavailable: {exc}"
    arr = np.asarray(flat_data, dtype=float).ravel()
    ref_img = None
    chunks = []
    for roi in roi_order:
        files = sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))
        if not files:
            return None, f"missing mask for {roi}"
        img = nib.load(str(files[0]))
        if ref_img is None:
            ref_img = img
        chunks.append(int((img.get_fdata() > 0).sum()))
    if sum(chunks) != arr.size:
        return None, f"ROI voxel count {sum(chunks)} does not match vector length {arr.size}"
    vol = np.zeros(ref_img.shape, dtype=float)
    cursor = 0
    for roi, n in zip(roi_order, chunks):
        img = nib.load(str(sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))[0]))
        mask_data = img.get_fdata() > 0
        vol[mask_data] = arr[cursor: cursor + n]
        cursor += n
    return nib.Nifti1Image(vol, ref_img.affine, ref_img.header), None

def _roi_sig_print(mask, roi_order, roi_dir, group):
    import nibabel as nib
    arr = np.asarray(mask, dtype=bool).ravel()
    cursor = 0
    rows = []
    for roi in roi_order:
        img = nib.load(str(sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))[0]))
        n = int((img.get_fdata() > 0).sum())
        selected = int(arr[cursor: cursor + n].sum())
        if selected > 0:
            rows.append({"Group": group, "ROI": roi, "ROI_voxels": n, "Selected_voxels": selected, "Percent_ROI": 100 * selected / n})
        cursor += n
    print(f"\nPermutation-significant voxels for {group}:")
    print(f"{'ROI Name':<25} | {'Voxels':<8} | {'% ROI'}")
    print("-" * 50)
    for row in sorted(rows, key=lambda x: x['Selected_voxels'], reverse=True):
        print(f"{row['ROI']:<25} | {row['Selected_voxels']:<8} | {row['Percent_ROI']:.2f}%")
    return pd.DataFrame(rows)

roi_order = _roi_vector_order_fear()
roi_plot_order = _roi_plot_order_fear()
roi_dir = _roi_dir_fear()
if roi_dir is None:
    display(Markdown("_Could not locate FearNetwork ROI masks for stage 11 glass brain._"))
else:
    try:
        from nilearn import plotting
        fig = plt.figure(figsize=(10, 8))
        axes = {"SAD": fig.add_subplot(2, 1, 1), "HC": fig.add_subplot(2, 1, 2)}
        roi_tables = []
        for group, ax in axes.items():
            mask = np.asarray(masks.get(group), dtype=bool) if group in masks else None
            score = np.asarray(scores.get(group), dtype=float) if group in scores else None
            if mask is None or score is None:
                ax.axis('off')
                ax.text(0.5, 0.5, f"{group}: missing mask or score", ha='center', va='center')
                continue
            values = np.where(mask, score, 0.0)
            img, err = _reconstruct_roi_map(values, roi_order, roi_dir)
            if img is None:
                ax.axis('off')
                ax.text(0.5, 0.5, f"{group}: {err}", ha='center', va='center', wrap=True)
                continue
            nonzero = np.abs(values[np.isfinite(values) & (values != 0)])
            threshold = float(np.min(nonzero)) if nonzero.size else 0.0
            vmax = float(np.percentile(nonzero, 99)) if nonzero.size else 1.0
            plotting.plot_glass_brain(img, display_mode='lyrz', colorbar=True, threshold=threshold, vmax=vmax, cmap='RdBu_r', plot_abs=False, black_bg=False, axes=ax, title=f"{group}: stage 11 permutation-significant voxels")
            roi_tables.append(_roi_sig_print(mask, roi_order, roi_dir, group))
        save_current_fig("section03_permutation_sig_voxels_glass_brain.png")
        plt.show()
        if roi_tables:
            roi_df = pd.concat(roi_tables, ignore_index=True)
            fig_roi, ax_roi = plt.subplots(figsize=(11, 5))
            sns.barplot(data=roi_df, x="ROI", y="Percent_ROI", hue="Group", order=[roi for roi in roi_plot_order if roi in set(roi_df["ROI"])], palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=ax_roi)
            for container in ax_roi.containers:
                labels = [f"{bar.get_height():.1f}%" if np.isfinite(bar.get_height()) and bar.get_height() > 0 else "" for bar in container]
                ax_roi.bar_label(container, labels=labels, padding=3, fontsize=9, rotation=90)
            ax_roi.set_ylim(0, max(roi_df["Percent_ROI"].max() * 1.22, 1))
            ax_roi.set_ylabel("Significant voxels (% of ROI)")
            ax_roi.set_xlabel("")
            ax_roi.set_title("ROI distribution of stage 11 permutation-significant voxels")
            ax_roi.tick_params(axis='x', rotation=60)
            save_current_fig("section03_permutation_sig_voxels_roi_percent.png")
            plt.show()
    except Exception as exc:
        display(Markdown(f"_Could not render stage 11 glass brain: {exc}_"))


## 4. Static Representational Topology
Statistics are listed first, followed by SAD and HC RDM heatmaps and distance summaries where the saved topology payload contains them.


In [ ]:
r12_payload = results.get("stage12_topology", {})
r12 = r12_payload.get("results_12", r12_payload) if isinstance(r12_payload, dict) else {}
RDM_CONDITIONS = r12.get("RDM_CONDITIONS") or r12_payload.get("RDM_CONDITIONS") or ["CS-", "CSS", "CSR"]
I_CS_MINUS, I_CSS, I_CSR = 0, 1, 2

def get_sig_star(p):
    if pd.isna(p): return "n/a"
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return "ns"

def extract_metrics(rdms):
    arr = np.asarray(rdms, dtype=float)
    return arr[:, I_CSR, I_CSS], arr[:, I_CSS, I_CS_MINUS]

def _stats_tuple(key, sad_vec, hc_vec):
    val = r12.get(key)
    if isinstance(val, (list, tuple, np.ndarray)) and len(val) >= 2:
        return float(val[0]), float(val[1])
    t, pval = stats.ttest_ind(sad_vec, hc_vec, equal_var=False, nan_policy='omit')
    return float(t), float(pval)

def _one_sample_p(prefix, key, vec):
    d = r12.get(prefix, {}) if isinstance(r12.get(prefix, {}), dict) else {}
    if key in d:
        return float(d[key])
    return float(stats.ttest_1samp(vec, 0, nan_policy='omit').pvalue)

def _print_metric_report(label, vec_a_sad, vec_a_hc, vec_b_sad, vec_b_hc, stat_a, stat_b, one_prefix):
    print("\n" + "="*110)
    print(label)
    print("="*110)
    rows = []
    for metric_label, sad_vec, hc_vec, stat_key, stat_pair in [
        ("Threat vs Safety", vec_a_sad, vec_a_hc, "a", stat_a),
        ("Safety vs Backgr", vec_b_sad, vec_b_hc, "b", stat_b),
    ]:
        p_sad = _one_sample_p(one_prefix, f"p_{stat_key}_sad", sad_vec)
        p_hc = _one_sample_p(one_prefix, f"p_{stat_key}_hc", hc_vec)
        t_diff, p_diff = stat_pair
        rows.extend([
            {"Metric": metric_label, "Group": "SAD", "Mean": np.nanmean(sad_vec), "p_vs_0": p_sad, "GroupDiff_t": t_diff, "GroupDiff_p": p_diff},
            {"Metric": metric_label, "Group": "HC", "Mean": np.nanmean(hc_vec), "p_vs_0": p_hc, "GroupDiff_t": np.nan, "GroupDiff_p": np.nan},
        ])
        print(f"{metric_label:<24} | SAD mean={np.nanmean(sad_vec):.6f}, p(vs0)={p_sad:.4f} | HC mean={np.nanmean(hc_vec):.6f}, p(vs0)={p_hc:.4f} | Group t={t_diff:.3f}, p={p_diff:.4f}")
    display_df(pd.DataFrame(rows), rows=10)

def plot_topology(rdms_sad, rdms_hc, vec_a_sad, vec_a_hc, vec_b_sad, vec_b_hc, p_a, p_b, p_a_sad_0, p_a_hc_0, p_b_sad_0, p_b_hc_0, title_suffix, filename):
    sns.set_context("poster")
    fig = plt.figure(figsize=(24, 8))
    gs = fig.add_gridspec(1, 3)
    ax1 = fig.add_subplot(gs[0, 0])
    sns.heatmap(np.nanmean(rdms_sad, axis=0), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1.2, xticklabels=RDM_CONDITIONS, yticklabels=RDM_CONDITIONS, ax=ax1, cbar=False)
    ax1.set_title(f"SAD Topology ({title_suffix})\n(n={len(rdms_sad)})")
    ax2 = fig.add_subplot(gs[0, 1])
    sns.heatmap(np.nanmean(rdms_hc, axis=0), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1.2, xticklabels=RDM_CONDITIONS, yticklabels=RDM_CONDITIONS, ax=ax2)
    ax2.set_title(f"HC Topology ({title_suffix})\n(n={len(rdms_hc)})")
    ax3 = fig.add_subplot(gs[0, 2])
    df_res = pd.DataFrame({
        'Group': ['SAD'] * len(vec_a_sad) + ['HC'] * len(vec_a_hc) + ['SAD'] * len(vec_b_sad) + ['HC'] * len(vec_b_hc),
        'Distance': np.concatenate([vec_a_sad, vec_a_hc, vec_b_sad, vec_b_hc]),
        'Metric': ['A: Threat Dist'] * len(vec_a_sad) + ['A: Threat Dist'] * len(vec_a_hc) + ['B: Safety Dist'] * len(vec_b_sad) + ['B: Safety Dist'] * len(vec_b_hc),
    })
    sns.violinplot(data=df_res, x='Metric', y='Distance', hue='Group', split=True, inner='quartile', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=ax3)
    ax3.set_title(f"Topological Metrics (Centroid | {title_suffix})")
    ax3.set_ylabel("Crossnobis Distance")
    y_max = df_res['Distance'].max()
    if p_a < 0.05:
        ax3.text(0, y_max + 0.05, f'* (p={p_a:.3f})', ha='center', fontsize=18)
    if p_b < 0.05:
        ax3.text(1, y_max + 0.05, f'* (p={p_b:.3f})', ha='center', fontsize=18)
    ax3.text(-0.2, -0.15, f"SAD: {get_sig_star(p_a_sad_0)}", transform=ax3.get_xaxis_transform(), ha='center', fontsize=14, color='#c44e52')
    ax3.text(0.2, -0.15, f"HC: {get_sig_star(p_a_hc_0)}", transform=ax3.get_xaxis_transform(), ha='center', fontsize=14, color='#4c72b0')
    ax3.text(0.8, -0.15, f"SAD: {get_sig_star(p_b_sad_0)}", transform=ax3.get_xaxis_transform(), ha='center', fontsize=14, color='#c44e52')
    ax3.text(1.2, -0.15, f"HC: {get_sig_star(p_b_hc_0)}", transform=ax3.get_xaxis_transform(), ha='center', fontsize=14, color='#4c72b0')
    save_current_fig(filename)
    plt.show()

display_df(flatten_result_dict(r12), rows=80)
plot_specs = [
    ("raw", "rdms_sad_raw", "rdms_hc_raw", "metric_a_stats_raw", "metric_b_stats_raw", "one_sample_stats_raw", "Important Features\n (p < 0.05, Raw)", "fearnetworkall_topology_raw.png"),
    ("z", "rdms_sad_z", "rdms_hc_z", "metric_a_stats_z", "metric_b_stats_z", "one_sample_stats_z", "Important Features\n (p < 0.05, Z-Scored)", "fearnetworkall_topology_z.png"),
    ("raw_pv", "rdms_sad_raw_pv", "rdms_hc_raw_pv", "metric_a_stats_raw_pv", "metric_b_stats_raw_pv", "one_sample_stats_raw_pv", "Important Features\n (p < 0.05, Raw, Per-Voxel)", "fearnetworkall_topology_raw_pv.png"),
]
for label, sad_key, hc_key, stat_a_key, stat_b_key, one_prefix, title_suffix, filename in plot_specs:
    if sad_key in r12 and hc_key in r12:
        rdms_sad = np.asarray(r12[sad_key], dtype=float)
        rdms_hc = np.asarray(r12[hc_key], dtype=float)
        vec_a_sad, vec_b_sad = extract_metrics(rdms_sad)
        vec_a_hc, vec_b_hc = extract_metrics(rdms_hc)
        stat_a = _stats_tuple(stat_a_key, vec_a_sad, vec_a_hc)
        stat_b = _stats_tuple(stat_b_key, vec_b_sad, vec_b_hc)
        _print_metric_report(f"TOPOLOGY REPORT: {label}", vec_a_sad, vec_a_hc, vec_b_sad, vec_b_hc, stat_a, stat_b, one_prefix)
        p_a_sad = _one_sample_p(one_prefix, "p_a_sad", vec_a_sad)
        p_a_hc = _one_sample_p(one_prefix, "p_a_hc", vec_a_hc)
        p_b_sad = _one_sample_p(one_prefix, "p_b_sad", vec_b_sad)
        p_b_hc = _one_sample_p(one_prefix, "p_b_hc", vec_b_hc)
        plot_topology(rdms_sad, rdms_hc, vec_a_sad, vec_a_hc, vec_b_sad, vec_b_hc, stat_a[1], stat_b[1], p_a_sad, p_a_hc, p_b_sad, p_b_hc, title_suffix, filename)


## 5. Dynamic Representational Drift
This section displays drift statistics and the drift figures saved by the script, then regenerates compact numeric summaries when possible.


In [ ]:
r13 = results.get("stage13_drift", {})
res13 = r13.get("results_13", r13) if isinstance(r13, dict) else {}
df_plot = res13.get("df_plot") if isinstance(res13, dict) else None
display_df(flatten_result_dict(res13), rows=60)
if isinstance(df_plot, pd.DataFrame) and not df_plot.empty:
    print(f"\n[Step 3] Analyzing {len(df_plot)} subject vectors...")
    display_df(df_plot, rows=25)
    sns.set_context("poster")
    fig, axes = plt.subplots(1, 3, figsize=(26, 8), gridspec_kw={'height_ratios': [1]})
    sns.barplot(data=df_plot, x='Condition', y='projection', hue='Group', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=axes[0], capsize=.1)
    axes[0].axhline(0, color='k', ls='--')
    axes[0].set_title("Neural Plasticity Magnitude\n(Projection)")
    axes[0].set_ylabel("Plasticity (au)")
    sns.barplot(data=df_plot, x='Condition', y='cosine', hue='Group', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=axes[1], capsize=.1)
    axes[1].axhline(0, color='k', ls='--')
    axes[1].set_title("Representational Fidelity\n(Cosine Similarity)")
    axes[1].set_ylabel("Fidelity (cos)")
    sns.scatterplot(data=df_plot, x='init_dist', y='projection', hue='Group', style='Condition', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=axes[2], s=150, alpha=0.8, edgecolor='black')
    axes[2].axhline(0, color='k', ls='--')
    axes[2].set_title("Plasticity vs. Initial Distance\n(Voxel Space)")
    axes[2].set_ylabel("Plasticity (au)")
    axes[2].grid(True, linestyle=':', alpha=0.5)
    print("\n--- Statistical Summary (p-values) ---")
    stat_rows = []
    for met in ['projection', 'cosine']:
        for cond in ['Safety', 'Threat']:
            d_s = df_plot[(df_plot['Condition'] == cond) & (df_plot['Group'] == 'SAD')][met].dropna()
            d_h = df_plot[(df_plot['Condition'] == cond) & (df_plot['Group'] == 'HC')][met].dropna()
            if len(d_s) > 1 and len(d_h) > 1:
                t, pval = stats.ttest_ind(d_s, d_h)
                sig = "*" if pval < 0.05 else "ns"
                print(f"  > Group Diff: {cond} {met} t={t:.3f}, p={pval:.4f} {sig}")
                stat_rows.append({'Condition': cond, 'Metric': met, 't': t, 'p': pval, 'sig': sig, 'mean_SAD': d_s.mean(), 'mean_HC': d_h.mean()})
    display_df(pd.DataFrame(stat_rows), rows=20)
    save_current_fig("fearnetworkall_stage11_drift_plots_row.png")
    plt.show()
else:
    display(Markdown("_No dynamic drift df_plot found._"))


## 6. Single-Trial Safety and Threat Trajectories
Safety and threat are plotted separately because they may use different trajectory scores. Trial-wise SAD versus HC Welch tests are reported when trial-level data are available.


In [ ]:
r14 = results.get("stage14_trajectories", {})
res132 = r14.get("results_13_2", r14) if isinstance(r14, dict) else {}
stats_safe = res132.get('stats_safe') if isinstance(res132, dict) else None
stats_threat = res132.get('stats_threat') if isinstance(res132, dict) else None
df_safe = res132.get('data_safe') if isinstance(res132, dict) else None
df_threat = res132.get('data_threat') if isinstance(res132, dict) else None
display(Markdown("**Safety trajectory statistics**")); display_df(stats_safe, rows=30)
display(Markdown("**Threat trajectory statistics**")); display_df(stats_threat, rows=30)

def _format_p_for_plot(p):
    if pd.isna(p):
        return "NA"
    p = float(p)
    if p < 0.001:
        return "<.001"
    return f"={p:.3f}".replace("0.", ".")

def _annotate_sig_group_diffs(ax, data_df, stats_df):
    if not isinstance(data_df, pd.DataFrame) or data_df.empty or not isinstance(stats_df, pd.DataFrame) or stats_df.empty:
        return
    if "Trial" not in stats_df.columns:
        return
    use_col = None
    label_prefix = "p"
    if "Diff_p_fdr" in stats_df.columns and (pd.to_numeric(stats_df["Diff_p_fdr"], errors="coerce") < 0.05).any():
        use_col = "Diff_p_fdr"
        label_prefix = "q"
    elif "Diff_p" in stats_df.columns:
        use_col = "Diff_p"
    if use_col is None:
        return
    sig = stats_df.loc[pd.to_numeric(stats_df[use_col], errors="coerce") < 0.05].copy()
    if sig.empty:
        return
    y_span = 3.0
    base_y = 1.72
    step = 0.09
    for i, (_, row) in enumerate(sig.iterrows()):
        trial = float(row["Trial"])
        pval = float(row[use_col])
        y = min(base_y + (i % 3) * step, 1.95)
        ax.plot([trial - 0.22, trial + 0.22], [y - 0.045, y - 0.045], color="black", lw=1.6, clip_on=True)
        ax.text(trial, y, f"{label_prefix}{_format_p_for_plot(pval)}", ha="center", va="bottom", fontsize=14, color="black", clip_on=True)
if isinstance(df_safe, pd.DataFrame) or isinstance(df_threat, pd.DataFrame):
    sns.set_context("poster")
    fig, axes = plt.subplots(1, 2, figsize=(22, 9), sharey=True)
    if isinstance(df_safe, pd.DataFrame) and not df_safe.empty:
        sns.lineplot(data=df_safe, x='trial', y='score', hue='Group', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, lw=3, marker='o', err_style='band', ax=axes[0])
        axes[0].set_title("A. Safety Trajectory\n(Target = CS-)")
        axes[0].set_ylabel("Similarity Score (0=Start, 1=Target)")
        axes[0].axhline(0, color='gray', ls='--', label='Start (Fear)')
        axes[0].axhline(1, color='#2ca02c', ls='-', lw=2, label='Target (CS-)')
        _annotate_sig_group_diffs(axes[0], df_safe, stats_safe)
        axes[0].set_ylim(-1, 2)
        axes[0].legend(loc='upper left')
    if isinstance(df_threat, pd.DataFrame) and not df_threat.empty:
        sns.lineplot(data=df_threat, x='trial', y='score', hue='Group', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, lw=3, marker='s', err_style='band', ax=axes[1])
        axes[1].set_title("B. Threat Maintenance\n(Target = Reinstated CSR)")
        axes[1].set_xlabel("Trial (Block Size: 1)")
        axes[1].axhline(0, color='gray', ls='--', label='Start (Ext Early)')
        axes[1].axhline(1, color='#d62728', ls='-', lw=2, label='Target (Reinstated CSR)')
        _annotate_sig_group_diffs(axes[1], df_threat, stats_threat)
        axes[1].set_ylim(-1, 2)
        axes[1].legend(loc='upper left')
    save_current_fig("fearnetworkall_stage12_trajectories_plot.png")
    plt.show()
else:
    display(Markdown("_No trajectory dataframes found._"))


## 7. Decision Boundary Characteristics
Decision-boundary and self-network uncertainty statistics are listed and plotted from saved results where available.


In [ ]:
r15 = results.get("stage15_decision", {})
res14 = r15.get("results_14_self", r15) if isinstance(r15, dict) else {}
df_sad_stats = res14.get('df_sad') if isinstance(res14, dict) else None
df_hc_stats = res14.get('df_hc') if isinstance(res14, dict) else None

def _bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    q = np.full(p.shape, np.nan, dtype=float)
    ok = np.isfinite(p)
    if not ok.any():
        return q
    idx = np.where(ok)[0]
    order = idx[np.argsort(p[ok])]
    ranked = p[order]
    m = len(ranked)
    adj = ranked * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    q[order] = np.minimum(adj, 1.0)
    return q

def _cohens_d_independent(a, b):
    a = pd.to_numeric(pd.Series(a), errors='coerce').dropna().to_numpy(dtype=float)
    b = pd.to_numeric(pd.Series(b), errors='coerce').dropna().to_numpy(dtype=float)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    pooled = np.sqrt(((len(a) - 1) * np.var(a, ddof=1) + (len(b) - 1) * np.var(b, ddof=1)) / (len(a) + len(b) - 2))
    return (np.mean(a) - np.mean(b)) / pooled if pooled > 0 else np.nan

def _p_text(p):
    if pd.isna(p):
        return 'NA'
    p = float(p)
    return '<.001' if p < 0.001 else f'={p:.3f}'.replace('0.', '.')

if isinstance(df_sad_stats, pd.DataFrame) and isinstance(df_hc_stats, pd.DataFrame):
    df_plot = pd.concat([df_sad_stats.assign(Group='SAD'), df_hc_stats.assign(Group='HC')], ignore_index=True)
    metric_specs = [
        ('entropy', 'Uncertainty\n(Entropy)'),
        ('kurtosis', 'Sharpness\n(Kurtosis)'),
        ('variance', 'Decision Probability\nVariance'),
        ('boundary_separation', 'CSS-CSR Boundary\nSeparation'),
        ('decision_margin_css', 'CSS Decision\nMargin'),
        ('decision_margin_all', 'Overall Decision\nMargin'),
        ('p_csr_css', 'P(CSR) for CSS\nSafety Cue'),
        ('p_csr_csr', 'P(CSR) for CSR\nThreat Cue'),
    ]
    rows = []
    for metric, label in metric_specs:
        if metric not in df_sad_stats.columns or metric not in df_hc_stats.columns:
            continue
        sad = pd.to_numeric(df_sad_stats[metric], errors='coerce').dropna()
        hc = pd.to_numeric(df_hc_stats[metric], errors='coerce').dropna()
        if len(sad) < 2 or len(hc) < 2:
            continue
        t_val, p_val = stats.ttest_ind(sad, hc, equal_var=False, nan_policy='omit')
        rows.append({
            'metric': metric,
            'label': label.replace('\n', ' '),
            'SAD_mean': float(sad.mean()),
            'HC_mean': float(hc.mean()),
            'SAD_sd': float(sad.std(ddof=1)),
            'HC_sd': float(hc.std(ddof=1)),
            't': float(t_val),
            'p': float(p_val),
            'cohens_d_SAD_minus_HC': float(_cohens_d_independent(sad, hc)),
        })
    stats_table = pd.DataFrame(rows)
    if not stats_table.empty:
        stats_table['p_fdr'] = _bh_fdr(stats_table['p'].to_numpy())
    all_p_sad = np.concatenate([np.asarray(p, dtype=float) for p in df_sad_stats['probabilities'].values if len(p) > 0]) if 'probabilities' in df_sad_stats.columns else np.array([])
    all_p_hc = np.concatenate([np.asarray(p, dtype=float) for p in df_hc_stats['probabilities'].values if len(p) > 0]) if 'probabilities' in df_hc_stats.columns else np.array([])
    if all_p_sad.size and all_p_hc.size:
        ks_stat, p_ks = stats.ks_2samp(all_p_sad, all_p_hc)
    else:
        ks_stat, p_ks = np.nan, np.nan
    print("\n" + "="*62)
    print("  DECISION-BOUNDARY SUMMARY: SAD vs HC")
    print("  Welch tests; p_fdr is BH-FDR across scalar boundary metrics")
    print("="*62)
    if not stats_table.empty:
        for _, row in stats_table.iterrows():
            print(f"  {row['metric']:<24} t = {row['t']:.4f}, p = {row['p']:.4f}, q = {row['p_fdr']:.4f}, d = {row['cohens_d_SAD_minus_HC']:.3f}")
    print(f"  probability distribution KS     D = {ks_stat:.4f}, p = {p_ks:.4f}")
    print("="*62 + "\n")
    display_df(stats_table, rows=30)
    display_df(pd.DataFrame([{'metric': 'probability_distribution_KS', 'D': ks_stat, 'p': p_ks}]), rows=5)
    sns.set_context("talk")
    fig, axes = plt.subplots(3, 3, figsize=(24, 18))
    axes_flat = axes.ravel()
    for ax, (metric, label) in zip(axes_flat, metric_specs):
        if metric not in df_plot.columns:
            ax.axis('off')
            continue
        sns.boxplot(data=df_plot, x='Group', y=metric, hue='Group', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, legend=False, ax=ax, width=0.55, fliersize=0)
        sns.stripplot(data=df_plot, x='Group', y=metric, hue='Group', palette={'SAD': '#8f2f34', 'HC': '#2e557f'}, legend=False, ax=ax, dodge=False, alpha=0.65, size=5)
        row = stats_table.loc[stats_table['metric'] == metric]
        if not row.empty:
            p_val = float(row.iloc[0]['p'])
            q_val = float(row.iloc[0]['p_fdr'])
            sig = '*' if q_val < 0.05 else ''
            ax.set_title(f"{label}\np{_p_text(p_val)}, q{_p_text(q_val)}{sig}")
        else:
            ax.set_title(label)
        ax.set_xlabel('')
    ax = axes_flat[len(metric_specs)]
    if all_p_sad.size and all_p_hc.size:
        sns.kdeplot(all_p_sad, fill=True, label='SAD', ax=ax, bw_adjust=1.0, color='#c44e52')
        sns.kdeplot(all_p_hc, fill=True, label='HC', ax=ax, bw_adjust=1.0, color='#4c72b0')
        ax.set_title(f"Neural Decision Density\nKS p{_p_text(p_ks)}")
        ax.set_xlabel("P(Threat / CSR)")
        ax.set_xlim(0, 1)
        ax.legend()
    else:
        ax.axis('off')
    fig.suptitle("Analysis 1.4 Decision Boundary Characteristics", y=1.01, fontsize=24)
    fig.tight_layout()
    save_current_fig("fearnetworkall_stage13_decision_extended_measures.png")
    plt.show()
else:
    display(Markdown("_Decision-stat dataframes not found._"))


## 8. Safety Restoration and Threat Discrimination
This section summarizes later-stage safety restoration and threat discrimination outputs.


In [ ]:
r16 = results.get("stage16_safety_threat", {})
res21 = r16.get("results_21_pv") or r16.get("results_21") or r16
df_topo_pv = res21.get("df") if isinstance(res21, dict) else None
p_safe = res21.get("p_safe", np.nan) if isinstance(res21, dict) else np.nan
p_threat = res21.get("p_threat", np.nan) if isinstance(res21, dict) else np.nan
display(Markdown("**Reference Figure: Analysis 2.1 Safety Restoration & Threat Discrimination**"))
display_df(flatten_result_dict(res21), rows=40)
if isinstance(df_topo_pv, pd.DataFrame) and not df_topo_pv.empty:
    display_df(df_topo_pv, rows=20)
    print(f"Safety restoration Group x Drug interaction p = {p_safe:.6g}")
    print(f"Threat discrimination Group x Drug interaction p = {p_threat:.6g}")
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    pal_group = {'SAD': '#c44e52', 'HC': '#4c72b0'}
    def plot_pv_metric(ax, y_col, title, ylabel, p_val):
        sns.pointplot(data=df_topo_pv, x='Drug', y=y_col, hue='Group', palette=pal_group, order=['Placebo', 'Oxytocin'], hue_order=['SAD', 'HC'], dodge=0.2, markers=['o', 's'], capsize=0.1, errorbar='se', ax=ax)
        sns.stripplot(data=df_topo_pv, x='Drug', y=y_col, hue='Group', palette=pal_group, order=['Placebo', 'Oxytocin'], hue_order=['SAD', 'HC'], dodge=True, alpha=0.3, jitter=True, legend=False, ax=ax)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        label = f"Interaction: p={p_val:.3f}" + ("*" if pd.notna(p_val) and p_val < 0.05 else "")
        ax.text(0.5, 0.95, label, transform=ax.transAxes, ha='center', color=('red' if pd.notna(p_val) and p_val < 0.05 else 'black'), fontweight=('bold' if pd.notna(p_val) and p_val < 0.05 else 'normal'))
    safety_col = 'Dist_Safety_PV' if 'Dist_Safety_PV' in df_topo_pv.columns else next((c for c in df_topo_pv.columns if 'Safety' in c), None)
    threat_col = 'Dist_Threat_PV' if 'Dist_Threat_PV' in df_topo_pv.columns else next((c for c in df_topo_pv.columns if 'Threat' in c), None)
    if safety_col:
        plot_pv_metric(axes[0], safety_col, "A. Safety Restoration (PV)\n(CSS vs CS-)", "PV Correlation Dist (Lower = Better)", p_safe)
    if threat_col:
        plot_pv_metric(axes[1], threat_col, "B. Threat Discrimination (PV)\n(CSR vs CSS)", "PV Correlation Dist (Higher = Better)", p_threat)
    save_current_fig("fearnetworkall_analysis21_safety_threat_pv.png")
    plt.show()
else:
    display(Markdown("_No Analysis 2.1 dataframe found._"))


## 9. Drift Efficiency for Safety and Threat Maintenance
This section reports drift-efficiency statistics and plots saved or reconstructed from the stage outputs.


In [ ]:
r18 = results.get("stage18_drift_efficiency", {})
res22 = r18.get("results_22") if isinstance(r18, dict) and isinstance(r18.get("results_22"), dict) else r18
df_drift = res22.get("df") if isinstance(res22, dict) else None
stats_results = res22.get("stats", {}) if isinstance(res22, dict) else {}
display(Markdown("**Reference Figure: Analysis 2.2 Drift Efficiency**"))
display_df(flatten_result_dict(res22), rows=40)
if isinstance(df_drift, pd.DataFrame) and not df_drift.empty:
    display_df(df_drift, rows=20)
    stats_table = pd.DataFrame([{"Domain_Metric": k, "interaction_p": v} for k, v in stats_results.items()])
    display_df(stats_table, rows=20)
    print("Interaction p-values:")
    for k, v in stats_results.items():
        print(f"  {k}: p={v:.6g}")
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    pal_group = {'SAD': '#c44e52', 'HC': '#4c72b0'}
    def _metric_col(metric):
        return metric if metric in df_drift.columns else metric.lower() if metric.lower() in df_drift.columns else None
    def plot_interaction(ax, domain, metric, p_val):
        mcol = _metric_col(metric)
        if mcol is None:
            ax.axis('off'); ax.set_title(f"Missing {metric}"); return
        sub = df_drift[df_drift['Domain'].astype(str).str.lower() == domain.lower()]
        sns.pointplot(data=sub, x='Drug', y=mcol, hue='Group', palette=pal_group, order=['Placebo', 'Oxytocin'], hue_order=['SAD', 'HC'], dodge=0.2, markers=['o', 's'], capsize=0.1, errorbar='se', ax=ax)
        ax.set_title(f"{domain}: {metric}\nInteraction p={p_val:.3f}")
    plot_interaction(axes[0,0], "Safety", "Cosine", stats_results.get("Safety_Cosine", np.nan))
    plot_interaction(axes[0,1], "Safety", "Projection", stats_results.get("Safety_Projection", np.nan))
    plot_interaction(axes[1,0], "Threat", "Cosine", stats_results.get("Threat_Cosine", np.nan))
    plot_interaction(axes[1,1], "Threat", "Projection", stats_results.get("Threat_Projection", np.nan))
    save_current_fig("fearnetworkall_analysis22_drift_efficiency.png")
    plt.show()
else:
    display(Markdown("_No drift-efficiency dataframe found._"))


## 10. Probabilistic Opening / Decision-Probability Extraction
This section summarizes probabilistic opening and decision-probability extraction outputs.


In [ ]:
r19 = results.get("stage19_prob_opening", {})
res23 = r19.get("results_23") if isinstance(r19, dict) and isinstance(r19.get("results_23"), dict) else r19
df_metrics = res23.get("df") if isinstance(res23, dict) else None
stats_results = dict(res23.get("stats", {})) if isinstance(res23, dict) and isinstance(res23.get("stats", {}), dict) else {}

# Match Section 7 decision-boundary measures when they are available in saved stage 15 outputs.
r15_for_opening = results.get("stage15_decision", {})
res14_for_opening = r15_for_opening.get("results_14_self", r15_for_opening) if isinstance(r15_for_opening, dict) else {}
df_sad_decision = res14_for_opening.get('df_sad') if isinstance(res14_for_opening, dict) else None
df_hc_decision = res14_for_opening.get('df_hc') if isinstance(res14_for_opening, dict) else None
section7_to_section10 = {
    'entropy': 'Entropy',
    'kurtosis': 'Kurtosis',
    'variance': 'Variance',
    'boundary_separation': 'Boundary Separation',
    'decision_margin_css': 'CSS Decision Margin',
    'decision_margin_all': 'Overall Decision Margin',
    'p_csr_css': 'P(CSR) CSS',
    'p_csr_csr': 'P(CSR) CSR',
}
measure_order = list(section7_to_section10.values())

def _interaction_p_lm(df, metric):
    sub = df[['Group', 'Drug', metric]].copy()
    sub[metric] = pd.to_numeric(sub[metric], errors='coerce')
    sub = sub.dropna()
    if sub['Group'].nunique() != 2 or sub['Drug'].nunique() != 2 or len(sub) < 8:
        return np.nan
    group_levels = sorted(sub['Group'].dropna().unique())
    drug_levels = sorted(sub['Drug'].dropna().unique())
    g = (sub['Group'] == group_levels[-1]).astype(float).to_numpy()
    d = (sub['Drug'] == drug_levels[-1]).astype(float).to_numpy()
    y = sub[metric].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(sub)), g, d, g * d])
    try:
        beta, residuals, rank, svals = np.linalg.lstsq(X, y, rcond=None)
        df_resid = len(y) - rank
        if df_resid <= 0:
            return np.nan
        rss = float(np.sum((y - X @ beta) ** 2))
        sigma2 = rss / df_resid
        cov = sigma2 * np.linalg.pinv(X.T @ X)
        se = float(np.sqrt(cov[3, 3]))
        if not np.isfinite(se) or se == 0:
            return np.nan
        t_val = float(beta[3] / se)
        return float(2 * stats.t.sf(abs(t_val), df_resid))
    except Exception:
        return np.nan

if isinstance(df_metrics, pd.DataFrame) and not df_metrics.empty:
    df_metrics = df_metrics.copy()
    native_to_display = {
        'P_CSR_CSS': 'P(CSR) CSS',
        'P_CSR_CSR': 'P(CSR) CSR',
        'Boundary_Separation': 'Boundary Separation',
        'Decision_Margin_CSS': 'CSS Decision Margin',
        'Decision_Margin_All': 'Overall Decision Margin',
    }
    for native_col, display_col in native_to_display.items():
        if native_col in df_metrics.columns and display_col not in df_metrics.columns:
            df_metrics[display_col] = df_metrics[native_col]
    if isinstance(df_sad_decision, pd.DataFrame) and isinstance(df_hc_decision, pd.DataFrame):
        decision_df = pd.concat([df_sad_decision.assign(Group='SAD'), df_hc_decision.assign(Group='HC')], ignore_index=True)
        rename_cols = {'sub': 'Subject'}
        rename_cols.update(section7_to_section10)
        decision_df = decision_df.rename(columns=rename_cols)
        merge_cols = ['Subject'] + [col for col in measure_order if col in decision_df.columns]
        # Keep Section 10 native Entropy/Kurtosis/Variance if present; add only missing/expanded measures from Section 7.
        add_cols = ['Subject'] + [col for col in merge_cols if col != 'Subject' and col not in df_metrics.columns]
        if len(add_cols) > 1:
            df_metrics = df_metrics.merge(decision_df[add_cols].drop_duplicates('Subject'), on='Subject', how='left')
    display(Markdown("**Reference Figure: Analysis 2.3 Probabilistic Opening / Decision-Probability Extraction**"))
    display(Markdown("Expanded to the same scalar decision-boundary measure set used in Section 7 when saved measures are available."))
    display_df(flatten_result_dict(res23), rows=40)
    display_df(df_metrics, rows=25)
    stat_rows = []
    for met in measure_order:
        if met not in df_metrics.columns:
            continue
        p_interaction = stats_results.get(met, _interaction_p_lm(df_metrics, met))
        stat_rows.append({'measure': met, 'group_x_drug_interaction_p': p_interaction, 'n_nonmissing': int(pd.to_numeric(df_metrics[met], errors='coerce').notna().sum())})
        print(f"{met} Group x Drug interaction p = {p_interaction:.6g}" if np.isfinite(p_interaction) else f"{met} Group x Drug interaction p = NA")
    stat_table = pd.DataFrame(stat_rows)
    display_df(stat_table, rows=30)
    plot_measures = [met for met in measure_order if met in df_metrics.columns]
    n_cols = 3
    n_rows = int(np.ceil(len(plot_measures) / n_cols)) if plot_measures else 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 6 * n_rows), squeeze=False)
    for ax, met in zip(axes.ravel(), plot_measures):
        pval_series = stat_table.loc[stat_table['measure'] == met, 'group_x_drug_interaction_p']
        pval = float(pval_series.iloc[0]) if len(pval_series) and pd.notna(pval_series.iloc[0]) else np.nan
        sns.boxplot(data=df_metrics, x='Group', y=met, hue='Drug', ax=ax)
        sns.stripplot(data=df_metrics, x='Group', y=met, hue='Drug', dodge=True, ax=ax, alpha=0.45, size=4, legend=False, color='black')
        ax.set_title(f"{met}\nGroup x Drug p={pval:.3f}" if np.isfinite(pval) else f"{met}\nGroup x Drug p=NA")
        ax.set_xlabel('')
        if ax.get_legend() is not None:
            ax.get_legend().set_title('Drug')
    for ax in axes.ravel()[len(plot_measures):]:
        ax.axis('off')
    fig.suptitle("Analysis 2.3 Probabilistic Opening: Section 7 Measure Set", y=1.01, fontsize=24)
    fig.tight_layout()
    save_current_fig("fearnetworkall_analysis23_probabilistic_opening_extended_measures.png")
    plt.show()
else:
    display(Markdown("_No probabilistic-opening dataframe found._"))


## 11. Spatial Re-Alignment
Spatial re-alignment statistics and figures are displayed here.


In [ ]:
r20 = results.get("stage20_realignment", {})
res24 = r20.get("results_24") if isinstance(r20, dict) and isinstance(r20.get("results_24"), dict) else r20
acc_plc = np.asarray(res24.get("acc_plc", []), dtype=float) if isinstance(res24, dict) else np.array([])
acc_oxt = np.asarray(res24.get("acc_oxt", []), dtype=float) if isinstance(res24, dict) else np.array([])
p_val = float(res24.get("p_val", np.nan)) if isinstance(res24, dict) else np.nan
display(Markdown("**Reference Figure: Analysis 2.4 Spatial Re-Alignment**"))
display_df(flatten_result_dict(res24), rows=40)
if acc_plc.size and acc_oxt.size:
    m_plc, m_oxt = np.nanmean(acc_plc), np.nanmean(acc_oxt)
    print(f"Accuracy (Train HC -> Test SAD-Placebo): {m_plc:.3f}; n={acc_plc.size}")
    print(f"Accuracy (Train HC -> Test SAD-Oxytocin): {m_oxt:.3f}; n={acc_oxt.size}")
    print(f"OXT > Placebo p = {p_val:.6g}")
    sig_label = "*" if p_val < 0.05 else "ns"
    fig, ax = plt.subplots(figsize=(10, 5))
    matrix_data = np.array([[m_plc, m_oxt]])
    annot_data = np.array([[f"{m_plc:.3f}", f"{m_oxt:.3f}\n({sig_label})"]])
    sns.heatmap(matrix_data, annot=annot_data, fmt="", cmap="RdBu_r", vmin=0.3, vmax=0.7, center=0.5, cbar=True, xticklabels=['Test: SAD-Placebo', 'Test: SAD-Oxytocin'], yticklabels=['Train: HC-Placebo (Anal 1.1)'], ax=ax)
    ax.set_title(f"Analysis 2.4: Spatial Re-Alignment\n(OXT vs PLC Improvement: p={p_val:.3f})")
    plt.yticks(rotation=0)
    save_current_fig("fearnetworkall_analysis24_spatial_realignment.png")
    plt.show()
else:
    display(Markdown("_No spatial realignment accuracy arrays found._"))


## 12. Reverse Cross-Decoding
Reverse cross-decoding statistics and figures are displayed here.


In [ ]:
r21 = results.get("stage21_reverse", {})
res25 = r21.get("results_25") if isinstance(r21, dict) and isinstance(r21.get("results_25"), dict) else r21
acc_plc = np.asarray(res25.get("acc_plc", []), dtype=float) if isinstance(res25, dict) else np.array([])
acc_oxt = np.asarray(res25.get("acc_oxt", []), dtype=float) if isinstance(res25, dict) else np.array([])
p_val = float(res25.get("p_val", np.nan)) if isinstance(res25, dict) else np.nan
display(Markdown("**Reference Figure: Analysis 2.5 Reverse Cross-Decoding**"))
display_df(flatten_result_dict(res25), rows=40)
if acc_plc.size and acc_oxt.size:
    m_plc, m_oxt = np.nanmean(acc_plc), np.nanmean(acc_oxt)
    print(f"Accuracy (Train SAD -> Test HC-Placebo): {m_plc:.3f}; n={acc_plc.size}")
    print(f"Accuracy (Train SAD -> Test HC-Oxytocin): {m_oxt:.3f}; n={acc_oxt.size}")
    print(f"OXT > Placebo p = {p_val:.6g}")
    sig_label = "*" if p_val < 0.05 else "ns"
    fig, ax = plt.subplots(figsize=(10, 5))
    matrix_data = np.array([[m_plc, m_oxt]])
    annot_data = np.array([[f"{m_plc:.3f}", f"{m_oxt:.3f}\n({sig_label})"]])
    sns.heatmap(matrix_data, annot=annot_data, fmt="", cmap="RdBu_r", vmin=0.3, vmax=0.7, center=0.5, cbar=True, xticklabels=['Test: HC-Placebo', 'Test: HC-Oxytocin'], yticklabels=['Train: SAD-Placebo (Anal 1.1)'], ax=ax)
    ax.set_title(f"Analysis 2.5: Reverse Cross-Decoding\n(OXT vs PLC Improvement: p={p_val:.3f})")
    plt.yticks(rotation=0)
    save_current_fig("fearnetworkall_analysis25_reverse_cross_decoding.png")
    plt.show()
else:
    display(Markdown("_No reverse cross-decoding accuracy arrays found._"))


## 13. Group-Wise Neural-Clinical Pearson Correlations
Pearson correlation tables and heatmaps are shown for group-wise neural-clinical associations.


In [ ]:
r27 = results.get("stage27_pearson", {})
r26 = results.get("stage26_master_merge", {})
df_master = r26.get("df_master_analysis") if isinstance(r26, dict) else None
df_res_grp = r27.get("df_res_grp") if isinstance(r27, dict) else None
neural_metrics = list(r27.get("neural_metrics", [])) if isinstance(r27, dict) else []
clinical_indices = list(r27.get("clinical_indices", [])) if isinstance(r27, dict) else []
if not neural_metrics and isinstance(df_res_grp, pd.DataFrame):
    neural_metrics = list(df_res_grp["Neural"].dropna().unique())
if not clinical_indices and isinstance(df_res_grp, pd.DataFrame):
    clinical_indices = list(df_res_grp["Clinical"].dropna().unique())
display(Markdown("**Reference Figures: Group-wise neural-clinical Pearson regression grids**"))
display_df(df_res_grp, rows=80)
if isinstance(df_master, pd.DataFrame) and isinstance(df_res_grp, pd.DataFrame) and not df_res_grp.empty:
    group_col = "Group" if "Group" in df_master.columns else "Group_x" if "Group_x" in df_master.columns else None
    print(f"{'Group':<6} | {'Neural Metric':<32} | {'Clinical':<28} | {'r':<7} | {'p':<10} | Sig")
    print("-" * 100)
    for _, row in df_res_grp.iterrows():
        print(f"{row['Group']:<6} | {row['Neural']:<32} | {row['Clinical']:<28} | {row['r']:<7.3f} | {row['p']:<10.5f} | {row['sig']}")
    plot_neural = neural_metrics[:]
    plot_clinical = clinical_indices[:]
    for n_m in plot_neural:
        if n_m not in df_master.columns:
            continue
        fig, axes = plt.subplots(1, len(plot_clinical), figsize=(max(8, 5.2 * len(plot_clinical)), 6))
        axes = np.atleast_1d(axes)
        for j, c_i in enumerate(plot_clinical):
            ax = axes[j]
            if c_i not in df_master.columns or group_col is None:
                ax.axis('off'); continue
            for grp in sorted(df_master[group_col].dropna().unique()):
                grp_data = df_master[df_master[group_col] == grp]
                stats_row = df_res_grp[(df_res_grp['Group'] == grp) & (df_res_grp['Neural'] == n_m) & (df_res_grp['Clinical'] == c_i)]
                label_str = str(grp)
                if not stats_row.empty:
                    sig = stats_row['sig'].values[0]
                    r = stats_row['r'].values[0]
                    label_str = f"{grp} (r={r:.2f}{'' if sig=='ns' else sig})"
                sns.regplot(data=grp_data, x=n_m, y=c_i, ax=ax, label=label_str, scatter_kws={'alpha': 0.45})
            ax.set_title(c_i.upper())
            ax.legend(fontsize=8, loc='best')
            sns.despine(ax=ax)
        plt.suptitle(f"Group-Wise Associations: {n_m.replace('_', ' ')}", fontsize=16, y=1.02)
        save_current_fig(f"fearnetworkall_pearson_grid_{n_m}.png")
        plt.show()
else:
    display(Markdown("_Pearson results or master dataframe not found._"))


## 14. Clinical Scores, Neural Indices, and Master Merge
The request listed section 14 without a label, so this notebook uses it as the audit point for clinical-score loading, neural-index extraction, and the merged analysis table used by the correlation/regression stages.


In [ ]:
for name in ["stage23_clinical_scores", "stage24_neural_indices", "stage26_master_merge"]:
    display(Markdown(f"**{name}**"))
    res = results.get(name, {})
    display_df(flatten_result_dict(res), rows=50)
    df = extract_first_dataframe(res, preferred=("df_master", "df_master_analysis", "df_scores", "df_indices", "df_neural"))
    if isinstance(df, pd.DataFrame):
        display_df(df, rows=20)
        maybe_plot_numeric_bars(df, name.replace("_", " "))


## 15. Group-Wise Partial Correlations Adjusted for Covariates
Partial correlation tables and heatmaps are shown when the saved stage contains adjusted association estimates.


In [ ]:
r26 = results.get("stage26_master_merge", {})
r28 = results.get("stage28_partial", {})
df_master = r26.get("df_master_analysis") if isinstance(r26, dict) else None
base_pearson = results.get("stage27_pearson", {})
neural_metrics = list(base_pearson.get("neural_metrics", [])) if isinstance(base_pearson, dict) else []
clinical_indices = list(base_pearson.get("clinical_indices", [])) if isinstance(base_pearson, dict) else []
covariates = ["demo_age"]
def _sig_star(p):
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return "ns"
def _partial_corr_resid(df, x, y, covars):
    cols = [x, y] + [c for c in covars if c in df.columns]
    data = df[cols].apply(pd.to_numeric, errors='coerce').dropna()
    if len(data) <= len(covars) + 2:
        return np.nan, np.nan, len(data)
    if covars and all(c in data.columns for c in covars):
        X = sm.add_constant(data[covars], has_constant='add')
        rx = sm.OLS(data[x], X).fit().resid
        ry = sm.OLS(data[y], X).fit().resid
    else:
        rx = data[x]
        ry = data[y]
    r, pval = stats.pearsonr(rx, ry)
    return r, pval, len(data)
display(Markdown("**Reference Figures: Group-wise partial correlations adjusted for covariates**"))
if isinstance(df_master, pd.DataFrame):
    group_col = "Group" if "Group" in df_master.columns else "Group_x" if "Group_x" in df_master.columns else None
    if not neural_metrics:
        neural_metrics = [c for c in df_master.columns if c.startswith('Neural_') and not c.endswith('_z')]
    if not clinical_indices:
        clinical_indices = [c for c in ['lsas_total','lsas_fear','lsas_avoid','dass_anxiety','dass_stress','ecr_total'] if c in df_master.columns]
    rows = []
    print(f"{'Group':<6} | {'Neural Metric':<32} | {'Clinical':<28} | {'r_adj':<7} | {'p':<10} | Sig | N")
    print("-" * 110)
    for grp in sorted(df_master[group_col].dropna().unique()) if group_col else []:
        df_grp = df_master[df_master[group_col] == grp]
        for n_m in neural_metrics:
            for c_i in clinical_indices:
                if n_m in df_grp.columns and c_i in df_grp.columns:
                    r, pval, n = _partial_corr_resid(df_grp, n_m, c_i, covariates)
                    if np.isfinite(r):
                        sig = _sig_star(pval)
                        print(f"{grp:<6} | {n_m:<32} | {c_i:<28} | {r:<7.3f} | {pval:<10.5f} | {sig:<3} | {n}")
                        rows.append({'Group': grp, 'Neural': n_m, 'Clinical': c_i, 'r': r, 'p': pval, 'sig': sig, 'N': n})
    df_partial = pd.DataFrame(rows)
    display_df(df_partial, rows=80)
    for n_m in neural_metrics:
        if n_m not in df_master.columns:
            continue
        fig, axes = plt.subplots(1, len(clinical_indices), figsize=(max(8, 5.2 * len(clinical_indices)), 6))
        axes = np.atleast_1d(axes)
        for j, c_i in enumerate(clinical_indices):
            ax = axes[j]
            if c_i not in df_master.columns:
                ax.axis('off'); continue
            for grp in sorted(df_master[group_col].dropna().unique()) if group_col else []:
                grp_data = df_master[df_master[group_col] == grp]
                stats_row = df_partial[(df_partial['Group'] == grp) & (df_partial['Neural'] == n_m) & (df_partial['Clinical'] == c_i)]
                label_str = str(grp)
                if not stats_row.empty:
                    sig = stats_row['sig'].values[0]
                    r = stats_row['r'].values[0]
                    label_str = f"{grp} (adj_r={r:.2f}{'' if sig=='ns' else sig})"
                sns.regplot(data=grp_data, x=n_m, y=c_i, ax=ax, label=label_str, scatter_kws={'alpha': 0.45})
            ax.set_title(c_i.upper())
            ax.legend(fontsize=8, loc='best')
            sns.despine(ax=ax)
        plt.suptitle(f"Group Associations Adjusted for Age: {n_m.replace('_', ' ')}", fontsize=16, y=1.02)
        save_current_fig(f"fearnetworkall_partial_grid_{n_m}.png")
        plt.show()
else:
    display(Markdown("_Master dataframe not found for partial correlations._"))


## 16. Outlier Removal and Z-Scoring
This section audits outlier removal and z-scoring for neural, clinical, and covariate variables.


In [ ]:
r26 = results.get("stage26_master_merge", {})
r29 = results.get("stage29_zscore_outlier", {})
df_master = r26.get("df_master_analysis") if isinstance(r26, dict) else None
base_pearson = results.get("stage27_pearson", {})
neural_metrics = list(base_pearson.get("neural_metrics", [])) if isinstance(base_pearson, dict) else []
clinical_indices = list(base_pearson.get("clinical_indices", [])) if isinstance(base_pearson, dict) else []
covariates = ['demo_age']
z_limit = float(r29.get('z_limit', 3.0)) if isinstance(r29, dict) else 3.0
display(Markdown("**Reference Audit: Outlier removal and final z-scoring**"))
if isinstance(df_master, pd.DataFrame):
    if not neural_metrics:
        neural_metrics = [c for c in df_master.columns if c.startswith('Neural_') and not c.endswith('_z')]
    if not clinical_indices:
        clinical_indices = [c for c in ['lsas_total','lsas_fear','lsas_avoid','dass_anxiety','dass_stress','ecr_total'] if c in df_master.columns]
    df_z = df_master.copy()
    rows = []
    for col in neural_metrics + clinical_indices + covariates:
        if col in df_z.columns:
            vals = pd.to_numeric(df_z[col], errors='coerce')
            init_z = (vals - vals.mean(skipna=True)) / vals.std(skipna=True, ddof=0)
            outlier_mask = init_z.abs() > z_limit
            n_removed = int(outlier_mask.sum())
            df_z.loc[outlier_mask, col] = np.nan
            vals2 = pd.to_numeric(df_z[col], errors='coerce')
            df_z[f'{col}_z'] = (vals2 - vals2.mean(skipna=True)) / vals2.std(skipna=True, ddof=0)
            rows.append({'variable': col, 'n_removed': n_removed, 'n_valid_after': int(vals2.notna().sum()), 'mean_after': float(vals2.mean(skipna=True)), 'sd_after': float(vals2.std(skipna=True, ddof=1))})
    summary = pd.DataFrame(rows)
    display_df(summary, rows=80)
    print(f"Outlier threshold: absolute z > {z_limit}")
    for _, row in summary.iterrows():
        if row['n_removed'] > 0:
            print(f"{row['variable']}: removed {int(row['n_removed'])} outliers")
    plot_cols = [f'{c}_z' for c in neural_metrics[:8] + clinical_indices[:8] if f'{c}_z' in df_z.columns]
    if plot_cols:
        fig, ax = plt.subplots(figsize=(max(10, 0.55 * len(plot_cols)), 5))
        sns.boxplot(data=df_z[plot_cols], ax=ax, color='#9C755F')
        ax.set_title('Z-scored neural, clinical, and covariate variables after outlier removal')
        ax.tick_params(axis='x', rotation=60)
        save_current_fig('fearnetworkall_outlier_zscore_boxplots.png')
        plt.show()
else:
    display(Markdown("_Master dataframe not found for z-scoring audit._"))


## 17. Z-Scored OLS / Regression Plots for Neural-Clinical Associations
This final section lists z-scored OLS models and plots regression coefficients or saved regression figures where available.


In [ ]:
r26 = results.get("stage26_master_merge", {})
base_pearson = results.get("stage27_pearson", {})
df_master = r26.get("df_master_analysis") if isinstance(r26, dict) else None
neural_metrics = list(base_pearson.get("neural_metrics", [])) if isinstance(base_pearson, dict) else []
clinical_indices = list(base_pearson.get("clinical_indices", [])) if isinstance(base_pearson, dict) else []
def _make_z_df(df, variables, z_limit=3.0):
    out = df.copy()
    for col in variables:
        if col in out.columns:
            vals = pd.to_numeric(out[col], errors='coerce')
            z0 = (vals - vals.mean(skipna=True)) / vals.std(skipna=True, ddof=0)
            out.loc[z0.abs() > z_limit, col] = np.nan
            vals2 = pd.to_numeric(out[col], errors='coerce')
            out[f'{col}_z'] = (vals2 - vals2.mean(skipna=True)) / vals2.std(skipna=True, ddof=0)
    return out
display(Markdown("**Reference Figures: Z-scored OLS/regression grids**"))
if isinstance(df_master, pd.DataFrame):
    group_col = "Group" if "Group" in df_master.columns else "Group_x" if "Group_x" in df_master.columns else None
    if not neural_metrics:
        neural_metrics = [c for c in df_master.columns if c.startswith('Neural_') and not c.endswith('_z')]
    if not clinical_indices:
        clinical_indices = [c for c in ['lsas_total','lsas_fear','lsas_avoid','dass_anxiety','dass_stress','ecr_total'] if c in df_master.columns]
    df_z = _make_z_df(df_master, neural_metrics + clinical_indices + ['demo_age'])
    neural_z = [f'{c}_z' for c in neural_metrics if f'{c}_z' in df_z.columns]
    clinical_z = [f'{c}_z' for c in clinical_indices if f'{c}_z' in df_z.columns]
    rows = []
    for n_m in neural_z:
        print(f"\n{'='*80}\nNEURAL PREDICTOR: {n_m.upper()}\n{'='*80}")
        fig, axes = plt.subplots(1, len(clinical_z), figsize=(max(8, 5.5 * len(clinical_z)), 6), sharey=True)
        axes = np.atleast_1d(axes)
        for i, c_i in enumerate(clinical_z):
            ax = axes[i]
            print(f"\n--- Clinical Outcome: {c_i.upper()} ---")
            for grp in sorted(df_z[group_col].dropna().unique()) if group_col else []:
                analysis_df = df_z[df_z[group_col] == grp][[n_m, c_i]].dropna()
                if len(analysis_df) > 5:
                    X = sm.add_constant(analysis_df[n_m])
                    y = analysis_df[c_i]
                    model = sm.OLS(y, X).fit()
                    beta = float(model.params[n_m])
                    tval = float(model.tvalues[n_m])
                    pval = float(model.pvalues[n_m])
                    sig_note = " *SIGNIFICANT*" if pval < 0.05 else ""
                    print(f"[{grp:<3}] N={len(analysis_df):<3} | Beta={beta:>6.3f} | t={tval:>6.2f} | p={pval:>6.4f}{sig_note}")
                    rows.append({'Group': grp, 'Neural_z': n_m, 'Clinical_z': c_i, 'N': len(analysis_df), 'beta': beta, 't': tval, 'p': pval})
                    sns.regplot(data=analysis_df, x=n_m, y=c_i, ax=ax, label=f"{grp} (p={pval:.3f})", scatter_kws={'alpha': 0.4}, line_kws={'lw': 3})
                else:
                    print(f"[{grp:<3}] Insufficient data (N < 5)")
            ax.set_title(c_i.replace('_z', '').upper())
            ax.set_xlabel(n_m)
            ax.set_ylabel(c_i)
            ax.axhline(0, color='gray', linestyle='--', alpha=0.3)
            ax.axvline(0, color='gray', linestyle='--', alpha=0.3)
            ax.legend(fontsize='small', loc='best')
        plt.suptitle(f"Brain-Behavior Associations: {n_m.replace('_z', '')} (Outliers Removed)", fontsize=16, y=1.02)
        sns.despine()
        save_current_fig(f"fearnetworkall_z_ols_grid_{n_m}.png")
        plt.show()
    df_ols = pd.DataFrame(rows)
    display_df(df_ols, rows=80)
else:
    display(Markdown("_Master dataframe not found for z-scored OLS grids._"))
